# 0. Introduction

Figure 1 : **(a)** The pipeline. **(b)** One individual, with illustrative genes — the GP writes the columns out of 89 terminals, ARD gives each column its own penalty and switches the useless ones off (dashed), and the fitness is PRESS plus a node price and a gene price.

그림 1 : **(a)** 전체 파이프라인. **(b)** 개체 하나(유전자는 예시) — GP가 터미널 89개로 열을 만들고, ARD가 열마다 벌점을 따로 매겨 쓸모없는 열을 끈다(점선). 적합도는 PRESS + 노드 가격 + 유전자 가격이다.

### **Summary**
A linear model can only weigh the columns it is handed.<br>
It will never write `GrLivArea × unit_price`, and it cannot drop a useless column on its own.<br>
So the work is split in two — **the GP picks the materials, ARD picks the coefficients.**

<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">Solver</th><th style="text-align:left">Decides</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>Multi-gene GP</b></td><td style="text-align:left">the materials — which columns exist, how many, and their shape</td></tr>
<tr><td style="text-align:left"><b>ARD</b></td><td style="text-align:left">the coefficients — the weight of each column, and its own penalty</td></tr>
</tbody></table>
</div>

An individual carries 1–45 expression trees (genes) over 89 terminals and 8 protected operators.<br>
Each gene becomes one column, and ARD reads the columns and returns a log price.<br>
The GP never sees the price itself — its fitness is how well ARD predicts with what it was given.

ARD (automatic relevance determination) is a ridge regression with one penalty per gene instead of one for all.<br>
Each penalty is set by maximising the evidence (MacKay 1992; Tipping 2001).<br>
A gene that explains nothing gets an ever larger penalty until its coefficient is exactly zero, while a useful gene is barely shrunk.<br>
A single ridge α had to squeeze good and useless genes alike.

The submission is the mean of the five fold models in log space.

### **Why `Utilities` is out**
Only one training house (Id 945) is not `AllPub`.<br>
In the test file every house is `AllPub` or blank, and blanks are filled as `AllPub`.<br>
All such a column can teach is how to point at Id 945, and the GP learned exactly that.

- In a run that kept the column, the gene `sqrt(Utilities)` fitted that one house with leverage 1.0000.
- Elsewhere the column served as a stray constant whose value depends on the fold — 68.2 where Id 945 is a training row, 4.0 where it is a validation row.

Nothing learned from it can ever reach a test row, so it is removed — 90 terminals become 89.

### **PRESS — no credit for memorising a row**
The previous run scored its genes with GCV, which approximates leave-one-out error by replacing each row's leverage with the average one.<br>
A gene that spikes on a single training row lets the decider fit that row exactly, and GCV counts the zero residual as a win.

PRESS (the predicted residual error sum of squares; Allen 1974) scores every training row with the model fitted without that row.<br>
For a linear decider it costs almost nothing — the leave-one-out residual is $e_i/(1-h_{ii})$, where $h_{ii}$ is the row's leverage.

- A gene that memorises a row exactly earns nothing — the row is still scored as if the gene were not there.
- A gene that nearly memorises a row (leverage close to 1) is punished — the small denominator blows the residual up.
- Every gene pays a price (next part), so a gene that earns nothing is a net loss and selection drops it.

Across our runs, the champions holding a training row with leverage above 0.9 fell from 72% under GCV to 0.1% under PRESS.<br>
ARD sets its penalties on all rows, so this PRESS holds them fixed when a row is left out.

### **How a run goes**
1. **Grow.** Every individual starts with one gene, and genes are free.<br>
The population grows until its mean gene count stops moving — less than 0.5 genes over 10 generations, judged between generations 30 and 150.
2. **Prune.** From then on every gene costs a price that rises linearly from 0 to 0.003 over 250 generations.<br>
The population sheds the genes that no longer pay their way, and the run ends when the price peaks, around generation 300.<br>
Every node also costs a fixed 1e-5, so dead weight cannot ride along for free.
3. **Pick.** The run traces a path from large models to small ones, and **the pooled-validation rule** picks one point on it.<br>
At each step after the growth stop it pools the five folds' validation errors (each fold scored on rows its model never trained on), smooths the curve over 11 steps, and returns every fold's champion at the same lowest step.<br>
Evolution itself never sees the validation rows — they only choose which snapshot to return.

Figure 2 shows the three phases in a full run of this setup.<br>
On Kaggle's four cores the full run takes about 7–9 hours, and `QUICK = True` walks the same path, small, in a few minutes.

Figure 2 : One full run of this setup on the same fold table, the five folds aligned at their growth stop (generations 49–69).<br>
**Top** — genes per individual (grey: each fold, black: the mean of five) and the gene price (red, right axis).<br>
**Bottom** — the pooled validation RMSE at each step after the growth stop (light: raw, dark: 11-step moving average). Its lowest point, +5, is the step returned for every fold.

그림 2 : 같은 셋업을 같은 fold 표로 끝까지 돌린 런 하나다. 다섯 fold를 각자의 다 자람 판정 세대(49\~69세대)에 맞춰 겹쳤다.<br>
**위** — 개체당 평균 유전자 수(회색: fold별, 검정: 다섯 fold 평균)와 유전자 가격(빨강, 오른쪽 축).<br>
**아래** — 판정 뒤 세대마다 다섯 fold의 valid 오차를 합친 통합 valid RMSE(옅은 선: 원래 값, 진한 선: 11칸 이동평균). 가장 낮은 +5가 다섯 fold 모두에 돌려줄 위치다.

### **Guards on rows the model has not seen**
- A gene's output on a validation or test row is clipped to the range it had on the training rows.
- Two genes can each stay inside their range and still combine into an extrapolation, so the prediction is also clipped to the fold's training log-price range.
- Id 524 and 1299 — 4,676 and 5,642 sq ft, sold for 184,750 and 160,000 dollars as partial sales — always stay on the training side, so they never decide which snapshot is returned, and they are left out of the price statistics (target-encoding levels, neighbourhood unit price).

### **What changed since the previous run**
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left"></th><th style="text-align:left">Previous run</th><th style="text-align:left">This notebook</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>Decider</b></td><td style="text-align:left">ridge, one α picked by GCV</td><td style="text-align:left">ARD, one α per gene set by the evidence</td></tr>
<tr><td style="text-align:left"><b>Fitness</b></td><td style="text-align:left">GCV + covariant parsimony</td><td style="text-align:left">PRESS + node price 1e-5 + gene price 0 → 0.003</td></tr>
<tr><td style="text-align:left"><b>Genes</b></td><td style="text-align:left">1–15, drawn at random at the start</td><td style="text-align:left">start at 1, grow up to 45</td></tr>
<tr><td style="text-align:left"><b>Breeding</b></td><td style="text-align:left">elite 1 + tournament 15</td><td style="text-align:left">elite 15% + tournament 15; crossover 90% / copy 10%; mutation 10%</td></tr>
<tr><td style="text-align:left"><b>Terminals</b></td><td style="text-align:left">90</td><td style="text-align:left">89 (<code>Utilities</code> out)</td></tr>
<tr><td style="text-align:left"><b>Id 524, 1299</b></td><td style="text-align:left">ordinary rows</td><td style="text-align:left">always on the train side, out of the price statistics</td></tr>
<tr><td style="text-align:left"><b>Unseen rows</b></td><td style="text-align:left">—</td><td style="text-align:left">gene outputs and predictions clipped</td></tr>
<tr><td style="text-align:left"><b>Returned model</b></td><td style="text-align:left">each fold's own best-validation snapshot (per-fold rule)</td><td style="text-align:left">one common step for all five folds (pooled validation)</td></tr>
<tr><td style="text-align:left"><b>Budget</b></td><td style="text-align:left">300 generations</td><td style="text-align:left">until the gene price peaks (about 300 generations), 8 h per fold at most</td></tr>
</tbody></table>
</div><br>

<hr>

### **GP가 재료를 고르고, ARD가 계수를 고른다**
선형 모델은 주어진 열의 비중만 정한다.<br>
`GrLivArea × 평당가` 같은 열을 새로 만들지 못하고, 쓸모없는 열을 스스로 빼지도 못한다.<br>
그래서 일을 둘로 나눈다 — 재료(열)는 GP가 고르고, 계수는 ARD가 고른다.

<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">누가</th><th style="text-align:left">무엇을 정하는가</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>다유전자 GP</b></td><td style="text-align:left">재료 — 어떤 열을 몇 개, 어떤 모양으로 만들지</td></tr>
<tr><td style="text-align:left"><b>ARD</b></td><td style="text-align:left">계수 — 열마다 줄 비중과 유전자별 벌점 α</td></tr>
</tbody></table>
</div>

GP의 적합도는 ARD가 그 재료로 가격을 얼마나 잘 맞히느냐로 정해진다.<br>
그래서 GP는 가격을 직접 맞히지 않고, ARD가 쓰기 좋은 재료를 만드는 쪽으로 진화한다.

개체 하나는 수식 나무 1\~45개다.<br>
나무는 터미널 89개(나무의 잎이 되는 입력 열)와 보호 연산자 8개(`add sub mul div log sqrt max min`)로 만든다 — 보호 연산자는 0으로 나누거나 음수에 로그를 씌워도 오류 없이 유한한 값을 돌려준다.<br>
나무 하나가 유전자 하나이고, 유전자 하나가 열 하나가 된다.

**ARD는 유전자마다 벌점을 따로 두는 릿지다.**<br>
릿지는 오차제곱합에 계수 제곱합의 α배를 벌점으로 더해 계수를 0 쪽으로 줄인다.<br>
α가 하나뿐이면 모든 유전자를 같은 세기로 누를 수밖에 없다.<br>
실제로 이전 런에서는 세대별 1등의 93\~97%가 후보 α 가운데 가장 작은 두 값(0.01 · 0.1)을 골랐다 — 벌점이 이렇게 약하면 아무 역할도 없는 유전자까지 거의 깎이지 않은 계수를 받는다.

ARD(자동 관련성 결정)는 각 계수에 '0에 가까울 것'이라는 믿음을 두고, 그 믿음의 세기를 유전자마다 따로 정한다.<br>
세기는 지금 가진 가격 데이터를 가장 그럴듯하게 설명하는 쪽으로 맞춘다 — 증거(evidence) 최대화다(MacKay 1992, Tipping 2001).<br>
쓸모없는 유전자는 벌점이 한없이 커져 계수가 정확히 0이 되고(꺼진다), 쓸모 있는 유전자는 거의 누르지 않는다.

제출 파일은 다섯 fold 모델의 예측을 log 공간에서 평균한 값이다.

### **선형 결합을 GP에게 맡기지 않는 이유**
처음에는 선형 결합까지 GP에게 맡겼는데, 세 가지 문제가 생겼다.

- **train조차 맞추지 못했다** : 같은 열을 릿지에 그대로 넣으면 valid가 0.12인데, GP는 train 오차가 0.14 아래로 내려가지 않았다. 세대를 세 배로 늘려도 거의 그대로였다.
- **상수가 살아남지 못했다** : log 가격은 절편 없이 맞출 수 없는데, 상수를 가진 개체는 0세대부터 드물었고 금방 사라졌다. 대신 개체군은 `KitchenAbvGr`처럼 거의 모든 행이 같은 값인 열을 상수처럼 썼고, 마지막 세대에는 개체군의 91%가 이 방법을 썼다.
- **범주형 원핫 열이 버려졌다** : 105열 중 실제로 쓰인 것은 10개 남짓이었다. Kronberger 외(GECCO 2018)도 GP가 명목형 지시 변수를 버리는 경향을 보고했다.

선형 결정자에는 절편이 있고 모든 열이 계수를 하나씩 받으므로, 선형 결합은 결정자에게 맡긴다.<br>
범주형 열은 원핫으로 넘기면 GP가 버리므로, GP에게 넘기기 전에 모양을 바꿔 둔다(3절의 세 갈래).

### **`Utilities`를 뺀 이유**
`Utilities`는 집에 들어오는 공공설비의 종류다.<br>
train 1,453행 중 1,452행이 `AllPub`(전기 · 가스 · 수도 · 하수 모두 있음)이고, Id 945 한 채만 `NoSeWa`(하수 · 수도 없음)다.<br>
test 1,459행은 모두 `AllPub`이거나 빈칸이고, 빈칸도 `AllPub`으로 채운다.<br>
이 열로 배울 수 있는 것은 Id 945를 짚어 내는 법 하나뿐이고, GP는 실제로 그것을 배웠다.

- **한 집을 외웠다** : `Utilities`를 남겨 둔 런에서 유전자 `sqrt(Utilities)`가 그 집을 레버리지 1.0000으로 맞췄다. 레버리지는 한 행의 예측값에서 그 행 자신의 정답이 차지하는 몫(0\~1)으로, 1이면 예측값이 곧 정답이다.
- **상수 대신 쓰였다** : 거의 모든 행의 값이 같아 상수 노릇을 했는데, 표준편차로 나눈 값이 fold마다 다르다 — Id 945가 train 행인 fold에서는 68.2, valid 행인 fold에서는 4.0이다. 같은 식이 fold마다 다른 뜻이 된다.

이렇게 배운 것은 test 어느 행에도 쓸 데가 없으므로 터미널에서 뺀다(90 → 89).<br>
드물어도 test에 같은 값이 나오는 열(도로 종류 `Street`, 수영장 관련 열)은 남긴다 — 그런 열을 외우는 문제는 아래의 PRESS가 막는다.

### **PRESS — 한 행을 통째로 외워서 얻는 점수를 없앤다**
이전 런은 적합도로 GCV를 썼다.<br>
GCV는 '한 행을 빼고 맞춘 모델이 그 행을 얼마나 맞히는가(한 행 빼기 오차)'를 근사하는데, 행마다 다른 레버리지를 평균값 하나로 바꿔 계산한다.<br>
그래서 어떤 유전자가 한 행에 딱 맞춰져 그 행의 잔차를 0으로 만들면, GCV는 그 0을 그대로 '잘 맞혔다'로 센다.

PRESS(predicted residual error sum of squares, Allen 1974)는 같은 한 행 빼기 오차를 근사 없이 잰다.<br>
선형 결정자라면 행을 하나씩 빼며 다시 풀 필요도 없다 — 한 번 맞춘 잔차 $e_i$와 레버리지 $h_{ii}$만 있으면 된다.

$$\text{PRESS} = \sqrt{\frac{1}{n}\sum_i \left(\frac{e_i}{1 - h_{ii}}\right)^2}, \qquad \text{GCV} = \frac{\text{RSS}/n}{(1 - \mathrm{df}/n)^2}$$

- **완전히 외운 유전자는 점수를 얻지 못한다** : 그 행은 그 유전자가 없을 때와 똑같이 채점된다.
- **거의 외운 유전자는 오히려 손해를 본다** : 레버리지가 1에 가까우면 분모 $1 - h_{ii}$가 작아져 그 행의 오차가 크게 부푼다.
- **점수를 못 내는 유전자는 밀려난다** : 유전자마다 가격이 붙으므로(아래 '런이 도는 방식') 가격만 내는 유전자는 선택에서 진다. 결과적으로 외우는 유전자에 벌점이 붙는다.

`Utilities`를 남겨 둔 런의 한 1등에서 `sqrt(Utilities)`가 Id 945를 외웠을 때, 두 기준은 이 유전자를 다음과 같이 봤다.

<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left"></th><th style="text-align:center">Id 945 학습 잔차</th><th style="text-align:center">레버리지</th><th style="text-align:center">한 행 빼기 잔차</th><th style="text-align:center">GCV</th><th style="text-align:center">PRESS</th></tr></thead>
<tbody>
<tr><td style="text-align:left">유전자 있음</td><td style="text-align:center">−0.0000</td><td style="text-align:center">1.0000</td><td style="text-align:center">−0.2485</td><td style="text-align:center">0.10145</td><td style="text-align:center">0.10243</td></tr>
<tr><td style="text-align:left">유전자 뺌</td><td style="text-align:center">−0.2381</td><td style="text-align:center">0.0417</td><td style="text-align:center">−0.2485</td><td style="text-align:center">0.10165</td><td style="text-align:center">0.10243</td></tr>
</tbody></table>
</div>

GCV는 이 유전자가 있으면 0.0002만큼 좋아진다고 보지만, PRESS는 있든 없든 0.10243으로 같다.<br>
효과도 쟀다 — ARD에 GCV를 쓴 런에서는 레버리지 0.9를 넘는 train 행을 가진 세대별 1등이 72%였고, PRESS를 쓰는 이 셋업의 런에서는 0.1%였다.

두 가지를 덧붙인다.

- **의미 있는 드문 값은 살아남는다** : 드문 값을 함께 가진 행이 여럿이고 서로를 예측할 수 있으면 PRESS도 점수를 준다. 수영장이 있는 집들이 그렇다.
- **벌점은 고정한다** : ARD의 벌점은 전체 train 행으로 정하고, 행을 하나 뺄 때마다 다시 고르지 않는다. 그래서 여기서 쓰는 PRESS는 엄밀히 말하면 '벌점을 고정한 한 행 빼기' 오차다.

### **런이 도는 방식 — 키우고, 깎고, 고른다**
1. **키운다** : 모든 개체는 유전자 1개로 시작하고, 처음에는 유전자를 늘려도 비용이 들지 않는다. 노드에는 하나당 1e-5를 늘 매겨, 아무 일도 하지 않는 노드가 공짜로 붙어 있지 못하게 한다.<br>
개체군의 평균 유전자 수가 10세대 동안 0.5개도 변하지 않으면 '다 자랐다'고 판정한다. 판정은 30세대부터 하고, 150세대까지 판정이 나지 않으면 150세대에서 다 자란 것으로 본다.
2. **깎는다** : 판정 다음 세대부터 유전자 하나의 가격을 250세대에 걸쳐 0에서 0.003까지 일정하게 올린다.<br>
개체군은 가격만큼 값을 못 하는 유전자부터 버리고, 가격이 최댓값에 닿는 300세대 안팎에서 런이 끝난다.
3. **고른다** : 그러면 큰 모델에서 작은 모델로 내려가는 길이 하나 생기고, 그 길의 어디서 멈출지는 valid 행으로 고른다.<br>
결정 나무의 비용-복잡도 가지치기와 같은 구조다. 그림 2가 실제 런의 세 단계다.

멈출 자리는 **통합 valid 최적**(영문의 pooled-validation rule)으로 고른다 — 다섯 fold에 똑같이 적용할 위치 하나를 고르는 규칙이다.

- **곡선** : 판정 뒤 p세대째마다(이때 유전자 가격은 fold끼리 같다) 다섯 fold의 '판정 + p세대' 1등이 낸 valid 제곱오차를 합쳐 곡선 하나를 만든다. valid 행은 그 행을 학습하지 않은 자기 fold 모델이 예측하므로, 합친 1,451행은 모두 처음 보는 행에 대한 예측이다.
- **선택** : 곡선을 11칸 이동평균한 뒤 가장 낮은 p를 고르고, 다섯 fold 모두 그 위치의 1등을 돌려준다.
- **왜 합치나** : fold마다 자기 valid 290행으로 따로 고르면 점수 하나의 표준오차(0.006\~0.016)가 후보끼리의 차이보다 커서 잡음을 고르게 된다. 합치면 판단에 쓰는 행이 다섯 배가 되고 결정은 한 번이면 된다. 교차검증으로 설정값을 고르는 교과서적인 절차와 같고(Hastie · Tibshirani · Friedman 2009, 7.10절), 여기서 고르는 설정값은 모델 크기, 곧 유전자 가격이다.

**진화는 valid 행을 보지 않는다.** valid 행은 세대마다 1등을 한 번 채점하고, 어느 세대를 돌려줄지 고르는 데만 쓴다.

### **처음 보는 행을 위한 안전장치**
- **유전자 클리핑** : valid · test 행에서는 유전자 출력을 그 유전자가 train 행에서 낸 최솟값과 최댓값 사이로 자른다. train 행의 값은 그대로이므로 ARD의 해는 바뀌지 않는다.
- **예측 클리핑** : 유전자 하나하나는 범위 안이어도, 둘이 합쳐져 학습 때 없던 조합으로 외삽할 수 있다 — 거의 같은 두 유전자에 부호가 반대인 큰 계수를 줘서 만든 '꺾인 선'이 그렇다. 그래서 예측값도 그 fold의 train 가격 범위로 자른다. valid 점수도 자른 예측으로 재므로, 돌려줄 세대를 고를 때와 제출할 때 같은 잣대를 쓴다.
- **이상치 두 채** : Id 524와 1299는 4,676 · 5,642sqft의 큰 집인데, 신축 분양(`Partial`) 거래로 184,750 · 160,000달러에 팔렸다. 데이터셋 저자도 이런 거래는 시장 가치를 반영하지 않을 가능성이 크다고 적었다(De Cock 2011). 그래도 train에서 빼지 않는다 — test에도 같은 유형의 집(Id 2550, 5,095sqft 분양)이 있고, 큰 집이 싸게 팔리는 경우를 배울 행이 이 두 채뿐이다. 대신 두 가지를 한다.
  - 모든 fold에서 train 쪽에 고정한다. valid 행에 들어가면 한 채가 그 fold valid 제곱오차의 3분의 1을 차지해, 실제로 반환 세대를 바꾼 적이 있다.
  - 가격 통계(타깃 인코딩의 수준별 평균, 동네 평균 단가)를 낼 때는 뺀다. 두 채의 평당가는 같은 동네(`Edwards`) 평균의 1/3이라, 넣으면 동네 단가가 끌려 내려간다.

PRESS · 클리핑 · `Utilities` 제거를 함께 넣은 뒤로, 10세대 이후 valid 오차가 0.2를 넘는 망가진 1등은 82개에서 0개가 됐다.

### **이전 런과 달라진 점**
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left"></th><th style="text-align:left">이전 런</th><th style="text-align:left">이 노트북</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>결정자</b></td><td style="text-align:left">릿지(α 하나, GCV로 선택)</td><td style="text-align:left">ARD(유전자마다 α, 증거 최대화로 결정)</td></tr>
<tr><td style="text-align:left"><b>적합도</b></td><td style="text-align:left">GCV + 공변 절약 벌점</td><td style="text-align:left">PRESS + 노드 가격 1e-5 + 유전자 가격 0 → 0.003</td></tr>
<tr><td style="text-align:left"><b>유전자 수</b></td><td style="text-align:left">1~15개, 처음에 무작위로 정함</td><td style="text-align:left">1개에서 시작해 최대 45개까지 늘어남</td></tr>
<tr><td style="text-align:left"><b>번식</b></td><td style="text-align:left">엘리트 1 + 토너먼트 15</td><td style="text-align:left">엘리트 15% + 토너먼트 15, 교차 90% / 복제 10%, 돌연변이 10%</td></tr>
<tr><td style="text-align:left"><b>터미널</b></td><td style="text-align:left">90</td><td style="text-align:left">89 (<code>Utilities</code> 제거)</td></tr>
<tr><td style="text-align:left"><b>Id 524 · 1299</b></td><td style="text-align:left">다른 행과 똑같이 취급</td><td style="text-align:left">train 쪽에 고정, 가격 통계에서 제외</td></tr>
<tr><td style="text-align:left"><b>처음 보는 행</b></td><td style="text-align:left">—</td><td style="text-align:left">유전자 출력과 예측값 클리핑</td></tr>
<tr><td style="text-align:left"><b>반환 모델</b></td><td style="text-align:left">fold마다 자기 valid가 가장 낮은 세대(fold별 최적)</td><td style="text-align:left">다섯 fold 공통 위치(통합 valid 최적)</td></tr>
<tr><td style="text-align:left"><b>예산</b></td><td style="text-align:left">300세대</td><td style="text-align:left">유전자 가격이 최댓값에 닿을 때까지(300세대 안팎), fold당 최대 8시간</td></tr>
</tbody></table>
</div><br>

# 1. Import

In [1]:
# Thread caps must be set BEFORE numpy is imported, or BLAS will already have spawned its pool.
# BLAS 스레드 상한은 **numpy import 전에** 걸어야 한다. 늦으면 이미 스레드 풀이 만들어져 있다.
import os

for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
           'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[_v] = '1'

import json, operator, random, resource, shutil, time, zipfile
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace

# Turn off all warnings
# 경고 off
import warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from deap import creator, base, gp, tools

# Belt and braces: if anything loaded numpy before this cell, the variables above came too late. threadpoolctl caps
# the BLAS that is actually loaded, and forked workers inherit the cap. One thread keeps every sum in the same order.
# 이 셀보다 먼저 numpy가 import됐다면 위 환경 변수는 이미 늦다. 그래서 threadpoolctl로 실제 불러온 BLAS를 1스레드로 묶고,
# fork한 워커도 이 설정을 물려받게 한다. 스레드가 하나여야 덧셈 순서가 늘 같아서 매번 같은 진화가 나온다.
from threadpoolctl import threadpool_info, threadpool_limits
threadpool_limits(1)

print('numpy  :', np.__version__)
print('pandas :', pd.__version__)
print('deap   : ok')

numpy  : 2.1.3
pandas : 2.3.3
deap   : ok


### Environment check / 환경 점검

In [2]:
# This pipeline is pure numpy on the CPU — a GPU has nothing to do here.
# 이 파이프라인은 CPU의 numpy만 쓴다. GPU는 쓰지 않는다.
N_CPU = os.cpu_count() or 1
RSS_MB = lambda: resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

print('cpu count       :', N_CPU)
print('thread cap      :', os.environ['OMP_NUM_THREADS'], '(per process)')
print('BLAS in effect  :', ', '.join(f"{i['internal_api']} {i['num_threads']} thread(s)" for i in threadpool_info()))
print(f'max RSS so far  : {RSS_MB():.0f} MB')
try:
    _mem = int(next(l.split()[1] for l in open('/proc/meminfo') if l.startswith('MemAvailable'))) / 1e6
    print(f'memory available: {_mem:.1f} GB')
except Exception:
    pass

# All five folds run at once, one worker each, even on four cores. The wall-clock cap is per fold, so a second
# wave would wait for the first and could run out of session time.
# 코어가 4개여도 fold 5개를 워커 5개로 동시에 돌린다. 시간 상한이 fold마다 걸리기 때문에, 두 번에 나눠 돌리면
# 두 번째 묶음이 첫 묶음을 기다리는 동안 세션 시간이 모자라게 된다.
PROCS = 5
print('worker processes:', PROCS)

cpu count       : 4
thread cap      : 1 (per process)
BLAS in effect  : openblas 1 thread(s), openblas 1 thread(s), openmp 1 thread(s)
max RSS so far  : 232 MB
memory available: 31.6 GB
worker processes: 5


### Directory setup / 경로 설정

In [3]:
# 경로 설정
if Path('/kaggle/input').exists():
    # when Kaggle
    ROOT_DIR = Path('/kaggle/working')
    COMP_DIR = next(p.parent for p in sorted(Path('/kaggle/input').rglob('train.csv'))
                    if (p.parent / 'test.csv').exists())      # 대회 데이터 / competition data
    SPLIT_DIR = next(p.parent for p in sorted(Path('/kaggle/input').rglob('folds.csv')))
    print('Environment : Kaggle')
else:
    # when local — point HP_COMP_DIR and HP_SPLIT_DIR at the inputs, or keep them next to the notebook
    # 로컬 — 입력 위치를 HP_COMP_DIR · HP_SPLIT_DIR로 주거나, 노트북 옆에 둔다
    ROOT_DIR = Path(os.environ.get('HP_KAGGLE_WORK', '.'))
    COMP_DIR = Path(os.environ.get('HP_COMP_DIR', '.'))
    SPLIT_DIR = Path(os.environ.get('HP_SPLIT_DIR', '.'))
    print('Environment : local')

OUTPUT_DIR = ROOT_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUB_PATH = ROOT_DIR / 'submission.csv'

print(f'ROOT_DIR   = {ROOT_DIR}')
print(f'COMP_DIR   = {COMP_DIR}')
print(f'SPLIT_DIR  = {SPLIT_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

Environment : Kaggle
ROOT_DIR   = /kaggle/working
COMP_DIR   = /kaggle/input/competitions/house-prices-advanced-regression-techniques
SPLIT_DIR  = /kaggle/input/datasets/illuminatep/hp-bundle
OUTPUT_DIR = /kaggle/working/outputs


### Check assets / 자산 검사

In [4]:
# 자산 검사
FOLDS_PATH = SPLIT_DIR / 'folds.csv'

_bad = []
for _f in ('train.csv', 'test.csv', 'sample_submission.csv'):
    if not (COMP_DIR / _f).exists():
        _bad.append(f'Missing {_f} ({COMP_DIR}) — attach the competition data')
if not FOLDS_PATH.exists():
    _bad.append(f'Missing folds.csv ({SPLIT_DIR}) — attach the split dataset')

assert not _bad, 'Asset check failed!!:\n  ' + '\n  '.join(_bad)
print('*** Asset check passed ***')

*** Asset check passed ***


### Flag : True for a quick wiring run, False for the full budget

In [5]:
# Quick run = True (a few minutes, proves the wiring) / Full budget = False
# 동작 확인 = True(몇 분, 같은 과정을 작게 한 번) · 본 런 = False(Kaggle 4코어에서 7~9시간)
QUICK = False
MAX_HOURS = 10.0        # wall-clock cap per fold. All five folds run at once, so this caps the whole run
                       # fold마다 거는 시간 상한. 다섯 fold가 함께 돌므로 런 전체의 상한이기도 하다
print(f'Mode: {"QUICK" if QUICK else "FULL"}')

Mode: FULL


### Set seed and budget / 시드와 예산

In [6]:
# It must be 42, since it is the ultimate answer to life, the universe and everything
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f'Seed = {SEED}')

# The material (section 3): one column out, two houses pinned to the train side and kept out of the price statistics.
# 재료(3절): 열 하나를 빼고, 집 두 채를 train 쪽에 고정하고 가격 통계에서 뺀다.
DROP_COLS = ('Utilities',)
PIN_IDS = (524, 1299)
STAT_EXCLUDE_IDS = (524, 1299)

CFG = dict(encoding='oof', decider='ard', fitness_metric='press', clip_genes=True,
           population=300 if QUICK else 10000,
           max_gen=60 if QUICK else 400,      # an upper bound only — the run ends when the gene price peaks
                                              # 상한일 뿐이다. 런은 유전자 가격이 최댓값에 닿으면 끝난다
           init_genes=(1, 1), genes=45, max_height=6,
           # Prices (section 4): a fixed price per node, and a gene price that rises once the population is grown.
           # 가격(4절): 노드 하나의 고정 가격, 그리고 개체군이 다 자란 뒤 오르는 유전자 가격.
           parsimony=1e-5, covariant_parsimony=False, size_measure='linear',
           parsimony_decay='sqrt_gen', parsimony_scale=1.0, parsimony_target=None,
           gene_price_max=0.003, gene_price_ramp=15 if QUICK else 250,
           grow_window=10, grow_flat=0.5, grow_min=10 if QUICK else 30, grow_max=20 if QUICK else 150,
           # Breeding (section 4) / 번식(4절)
           elite_frac=0.15, tournament=15, p_crossover=0.9, p_cross_hi=0.2, hi_cross_rate=0.5,
           p_mutation=0.1, mut_mix_early=(0.5, 0.1, 0.4), mut_mix_late=(0.1, 0.1, 0.8), mut_early_gens=10,
           return_smooth=11,                  # 통합 valid 최적의 이동평균 창(5절)
           alpha_theta=0.0, lm_early=0, lm_early_gens=0, lm_late=0,     # theta 정련(LM)은 쓰지 않는다
           seed=SEED, max_seconds=MAX_HOURS * 3600,
           # Per-generation records for the figures / 그림에 쓸 세대별 기록
           ckpt_top=5,        # 세대마다 남길 상위 개체 수. 1등은 통합 valid 최적의 후보도 된다
           census_top=50,     # 세대마다 남길 '가장 흔한 유전자식' 개수
           lineage=1)         # 개체군 전부의 부모-자식 간선 (0이면 끈다)

print(f'pop {CFG["population"]:,} · genes {CFG["init_genes"][0]} -> up to {CFG["genes"]} · '
      f'decider ARD · fitness PRESS + {CFG["parsimony"]:g} x nodes + gene price x genes')
print(f'gene price 0 until the population is grown (judged at generation {CFG["grow_min"]}~{CFG["grow_max"]}), '
      f'then 0 -> {CFG["gene_price_max"]} over {CFG["gene_price_ramp"]} generations · at most {CFG["max_gen"]} generations')
print(f'terminals without {list(DROP_COLS)} · pinned to the train side {list(PIN_IDS)}')
print(f'wall-clock cap per fold: {CFG["max_seconds"] / 3600:.1f} h')
print(f'records per generation: top {CFG["ckpt_top"]} individuals · census top {CFG["census_top"]}'
      f' · lineage {"on" if CFG["lineage"] else "off"}')

Seed = 42
pop 10,000 · genes 1 -> up to 45 · decider ARD · fitness PRESS + 1e-05 x nodes + gene price x genes
gene price 0 until the population is grown (judged at generation 30~150), then 0 -> 0.003 over 250 generations · at most 400 generations
terminals without ['Utilities'] · pinned to the train side [524, 1299]
wall-clock cap per fold: 10.0 h
records per generation: top 5 individuals · census top 50 · lineage on


# 2. Load the split dataset / 데이터셋 불러오기

입력 데이터셋에는 `Id, fold` 두 열짜리 fold 표 하나만 있다.<br>
행은 대회 `train.csv`에서 Id로 찾아온다. 이전 런과 같은 표다.

학습에는 쓸 수 있는 행을 모두 쓴다 — 모순된 7행을 지운 1,453행이고, 따로 떼어 두는 holdout은 없다.<br>
점수는 리더보드에서 읽는다.

- **valid 행** : fold마다 290\~291행. 진화에는 쓰지 않고, 5절의 통합 valid 최적이 돌려줄 세대를 고를 때만 쓴다.
- **이상치 두 채** : Id 524 · 1299는 어느 fold에서도 valid 행이 되지 않는다(3절).

In [7]:
# The fold table is the only thing shared — the rows come from the competition csv.
# 공유하는 것은 fold 표뿐이다. 행은 대회 csv에서 가져온다.
NA_OPTS_LOAD = dict(keep_default_na=False, na_values=['NA'])    # 'NA'만 결측으로 / only 'NA' is missing

train_csv = pd.read_csv(COMP_DIR / 'train.csv', **NA_OPTS_LOAD)
test_csv = pd.read_csv(COMP_DIR / 'test.csv', **NA_OPTS_LOAD)
fold_tbl = pd.read_csv(FOLDS_PATH)

assert list(fold_tbl.columns) == ['Id', 'fold'], f'fold table columns {list(fold_tbl.columns)}!!'
assert fold_tbl.Id.is_unique, 'duplicated Id in the fold table!!'
assert fold_tbl.Id.isin(train_csv.Id).all(), \
    f'{int((~fold_tbl.Id.isin(train_csv.Id)).sum())} fold ids are not in the competition train.csv!!'

FOLD_OF = dict(zip(fold_tbl.Id, fold_tbl.fold))
print(f'train.csv {len(train_csv):,} rows · test.csv {len(test_csv):,} rows')
print(f'fold table {len(fold_tbl):,} rows · sizes {np.bincount(fold_tbl.fold).tolist()}')

train.csv 1,460 rows · test.csv 1,459 rows
fold table 1,453 rows · sizes [291, 291, 291, 290, 290]


### Id sum check / Id 합 검증

In [8]:
# The fold table must be the competition train.csv minus the rows that contradict themselves.
# fold 표는 대회 train.csv에서 모순된 행만 뺀 것이어야 한다.
_extra = sorted(int(i) for i in set(train_csv.Id) - set(fold_tbl.Id))

print(f'{"item":<12}{"rows":>8}{"id sum":>14}')
for _k, _df in (('train.csv', train_csv), ('test.csv', test_csv), ('fold table', fold_tbl)):
    print(f'{_k:<12}{len(_df):>8,}{int(_df.Id.sum()):>14,}')

assert set(fold_tbl.Id) <= set(train_csv.Id), 'the fold table has ids that train.csv does not!!'
assert len(_extra) == 7, f'train.csv has {len(_extra)} rows outside the fold table, expected 7!!'
assert int(train_csv.Id.sum()) - int(fold_tbl.Id.sum()) == sum(_extra), 'id sum does not add up!!'

print(f'\n*** Id check passed ***')
print(f'    fold table = train.csv minus the 7 contradictory rows {_extra}')
print(f'    id sum gap {int(train_csv.Id.sum()) - int(fold_tbl.Id.sum()):,} = sum of those 7 ids')

item            rows        id sum
train.csv      1,460     1,066,530
test.csv       1,459     3,195,210
fold table     1,453     1,059,333

*** Id check passed ***
    fold table = train.csv minus the 7 contradictory rows [625, 689, 774, 1231, 1242, 1301, 1335]
    id sum gap 7,197 = sum of those 7 ids


# 3. Preprocessing / 전처리

원본 csv는 다섯 단계를 거쳐 수치형 터미널 89개가 된다.

1. **모순 처리** : train 7행은 값끼리 맞지 않는다(지하실 유형은 '없음'인데 지하실 면적은 있는 식). 이 7행은 지우고, 6칸은 값을 바로잡는다.
2. **결측 채우기** : `NA`는 어떤 열에서는 '없음', 어떤 열에서는 '모름'이다. 두 경우를 다르게 채운다.
3. **범주형 인코딩** : 글자 열 43개를 세 갈래로 나눠 숫자로 바꾼다. 그중 `Utilities`는 터미널에서 뺀다.
4. **누출 통제** : 정답(가격)을 쓰는 통계는 모두 그 fold의 train 행만으로 계산한다. 이상치 두 채는 train 쪽에 두되, 가격 통계에서는 뺀다.
5. **조립** : 가공한 78열 + OOF 타깃 인코딩 11열 = 89열.

## Rows — contradictions and missing values / 행 — 모순과 결측

### Fix contradictions and fill missing / 모순 처리와 결측 채우기

In [9]:
# Stage 1-2 — remove self-contradicting rows, then fill the two kinds of NA.
# 1~2단계 — 값끼리 모순되는 행을 지우고, 뜻이 다른 두 종류의 NA를 각각 다르게 채운다.
#   NA_IS_NONE   : 'NA'가 '없음'이라는 뜻인 열 (지하실 없음, 차고 없음 ...)
#   FIXED_FILL   : 'NA'가 '모름'이라 정해 둔 값으로 채우는 열
#   PAIR         : 한쪽이 없으면 다른 쪽도 없어야 하는 짝 (지하실 유형 ↔ 면적)


# 'NA'만 결측으로 읽는다. MasVnrType의 'None'은 값이다.
NA_OPTS = dict(keep_default_na=False, na_values=['NA'])


# Seven training rows contradict themselves (a basement area with no basement type). They go out.
# train 데이터 중 7행은 값끼리 모순된다(지하실 유형은 없는데 면적은 있는 식). 이 행들은 지운다.
TRAIN_DROP_IDS = (625, 689, 774, 1231, 1242, 1301, 1335)

TEST_YEAR_FIX_IDS = (1877, 2550, 2593)

TEST_MASVNR_FIX_IDS = (1670, 2453, 2320)

BSMT_CATS = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

BSMT_NUMS = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath']

GARAGE_CATS = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']

GARAGE_NUMS = ['GarageYrBlt', 'GarageCars', 'GarageArea']

PAIR = {**{c: ('TotalBsmtSF', 'NA') for c in BSMT_CATS},
        **{c: ('GarageArea', 'NA') for c in GARAGE_CATS},
        'FireplaceQu': ('Fireplaces', 'NA'),
        'PoolQC': ('PoolArea', 'NA'),
        'MasVnrType': ('MasVnrArea', 'None')}


# 'NA' means "none" in these columns and "unknown" elsewhere. The two cases are filled differently.
# 이 열들에서 'NA'는 '없음'이고, 다른 열에서는 '모름'이다. 두 경우를 다르게 채운다.
NA_IS_NONE = ['Alley', 'Fence', 'MiscFeature']


# NA가 '모름'인 열과 그 열을 채울 값.
FIXED_FILL = {'MSZoning': 'RL', 'Utilities': 'AllPub', 'Exterior1st': 'VinylSd', 'Exterior2nd': 'VinylSd',
              'KitchenQual': 'TA', 'Functional': 'Typ', 'SaleType': 'WD', 'Electrical': 'SBrkr'}


# 면적이 0보다 큰데 유형이 NA인 행에 넣을 값.
MASVNR_TYPE_WITH_AREA = 'BrkFace'

def numeric_columns(df):
    return [c for c in df.columns if c not in ('Id', 'SalePrice') and pd.api.types.is_numeric_dtype(df[c])]


# Fix what can be fixed, drop what cannot. Test rows are never dropped — the submission needs all 1,459.
# 고칠 수 있으면 고치고 못 고치면 버린다. test 행은 절대 버리지 않는다. 제출에 1,459행이 다 필요하다.
def fix_contradictions(train, test):
    train = train[~train['Id'].isin(TRAIN_DROP_IDS)].reset_index(drop=True)
    test = test.copy()
    changes = []

    def change(mask, col, new, rule):
        for idx in test.index[mask]:
            value = new[idx] if isinstance(new, pd.Series) else new
            if test.at[idx, col] != value:
                changes.append(dict(Id=int(test.at[idx, 'Id']), column=col, before=test.at[idx, col], after=value,
                                    rule=rule))
            test.at[idx, col] = value

    change(test['GarageYrBlt'] == 2207, 'GarageYrBlt', 2007, 'GarageYrBlt 2207 → 2007')
    change(test['YearRemodAdd'] < test['YearBuilt'], 'YearRemodAdd', test['YearBuilt'], 'YearRemodAdd < YearBuilt → YearBuilt')
    change(test['YrSold'] < test['YearBuilt'], 'YrSold', test['YearBuilt'], 'YrSold < YearBuilt → YearBuilt')
    area_one = test['MasVnrArea'] == 1
    change(area_one, 'MasVnrType', 'None', '마감재 면적 1 → 0·None')
    change(area_one, 'MasVnrArea', 0.0, '마감재 면적 1 → 0·None')
    change((test['MasVnrType'] == 'None') & (test['MasVnrArea'] > 1), 'MasVnrType', 'BrkFace', 'None + 면적 > 1 → BrkFace')
    has_type = test['MasVnrType'].notna() & (test['MasVnrType'] != 'None')
    change(has_type & (test['MasVnrArea'] == 0), 'MasVnrType', 'None', '종류 있음 + 면적 0 → None')
    return train, test, pd.DataFrame(changes, columns=['Id', 'column', 'before', 'after', 'rule'])

def _mode(values):
    modes = values.mode()
    return modes.iloc[0], list(modes)


# Paired columns decide whether an NA is "none" or "unknown": if the partner is 0 it is none, else unknown.
# 짝 컬럼으로 NA의 뜻을 가린다. 짝이 0이면 없음, 0보다 크면 모름(최빈값으로 채운다).
def fill_missing(train, test):
    pool = pd.concat([train, test], ignore_index=True)
    nb_lotfrontage = pool.groupby('Neighborhood')['LotFrontage'].median()
    fills, out = [], []

    for name, df in (('train', train), ('test', test)):
        df = df.copy()

        def fill(mask, col, value, rule):
            for idx in df.index[mask]:
                v = value[idx] if isinstance(value, pd.Series) else value
                fills.append(dict(dataset=name, Id=int(df.at[idx, 'Id']), column=col, value=v, rule=rule))
                df.at[idx, col] = v

        no_bsmt = df[BSMT_CATS].isna().all(axis=1) & (df['TotalBsmtSF'].isna() | (df['TotalBsmtSF'] == 0))
        for col in BSMT_NUMS:
            fill(no_bsmt & df[col].isna(), col, 0.0, '시설 없음 → 0')
        fill(df['GarageType'].isna() & df['GarageYrBlt'].isna(), 'GarageYrBlt', df['YearBuilt'], '차고 없음 → YearBuilt')
        for col in GARAGE_NUMS:
            fill(df['GarageType'].notna() & df[col].isna(), col, pool[col].median(), '차고 있음 → 중앙값')
        fill(df['LotFrontage'].isna(), 'LotFrontage', df['Neighborhood'].map(nb_lotfrontage), '지역별 중앙값')
        for col in numeric_columns(df):
            fill(df[col].isna(), col, pool[col].median(), '중앙값')

        for col, value in FIXED_FILL.items():
            fill(df[col].isna(), col, value, '정해 둔 값')
        for col in NA_IS_NONE:
            fill(df[col].isna(), col, 'NA', '없음')
        for col, (pair, none_value) in PAIR.items():
            fill(df[col].isna() & (df[pair] == 0), col, none_value, f'{pair} = 0 → 없음')
            missing = df[col].isna() & (df[pair] > 0)
            if not missing.any():
                continue
            if col == 'MasVnrType':
                fill(missing, col, MASVNR_TYPE_WITH_AREA, f'{pair} > 0 → BrkFace (면적이 있는데 유형이 비었다)')
                continue
            mode, modes = _mode(pool[col])
            tie = f' (동률 {"·".join(modes)} → {mode})' if len(modes) > 1 else ''
            fill(missing, col, mode, f'{pair} > 0 → 최빈값{tie}')
        out.append(df)

    fills = pd.DataFrame(fills, columns=['dataset', 'Id', 'column', 'value', 'rule'])
    return out[0], out[1], fills

## Columns — categorical encoding / 열 — 범주형 인코딩

### Three routes for 43 string columns / 글자 열 43개의 세 갈래

범주형 피쳐에 명확한 순서가 있으면 그것을 사용하고, 순서가 없는 경우 OOF 타깃 인코딩을 사용한다.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">갈래</th><th style="text-align:center">열 수</th><th style="text-align:left">이유</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>순서형 정수</b></td><td style="text-align:center">23</td><td style="text-align:left">누구나 납득할 만한 순서가 있다(<code>Ex &gt; Gd &gt; TA &gt; Fa &gt; Po</code>). 한 열이면 되고, 추정 잡음도 누출도 없다</td></tr>
<tr><td style="text-align:left"><b>파생 그룹</b></td><td style="text-align:center">9 → 21</td><td style="text-align:left">수준이 너무 많아서, 도메인 지식으로 묶는 편이 낫다(외장재 계열, 동네 평당가)</td></tr>
<tr><td style="text-align:left"><b>타깃 인코딩</b></td><td style="text-align:center">11</td><td style="text-align:left">납득할 만한 순서가 없으니, 정답(가격)으로 순서를 매긴다</td></tr>
</tbody></table>
</div>

순서형 23개까지 타깃 인코딩으로 바꾸면, 이미 아는 순서를 버리고 그 자리를 추정 잡음으로 채우는 셈이다.

In [10]:
# Stage 3 — one route per column, chosen by what the column's own structure allows.
# 3단계 — 열마다 그 열의 구조에 맞는 방식으로 인코딩한다.
#   ORDINAL   : 품질 등급을 정수 순서로 바꾼다(Ex=5 ... Po=1). Utilities도 여기서 인코딩하지만 설계행렬에서는 뺀다(build의 drop_cols)
#   DECOMPOSE : 한 열을 뜻이 다른 여러 0/1 열로 분해 (HouseStyle → 층수·반층·상층마감)
#   EXTERIOR  : 외장재 15종 → 재료 계열 5개 + '혼합' 표시
#   ONEHOT    : 순서가 없는 9열. 여기서 만든 0/1 열은 버리고 타깃 인코딩으로 대체한다(GarageType·MasVnrType까지 11열)
#   UNIT_PRICE: 동네별 평당가. 유일하게 가격을 쓰는 열이고, train 행에서는 자기 행을 빼고 계산한다

QUALITY = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}


# 23 columns have a defensible order, so they become integers. One column, no estimation noise, no leakage.
# 23개 열은 누구나 납득할 순서가 있어 정수로 바꾼다. 한 열로 충분하고 추정 잡음도 누출도 없다.
ORDINAL = {
    'ExterQual': dict(QUALITY),
    'ExterCond': dict(QUALITY),
    'BsmtQual': {'NA': 0, **QUALITY},
    'BsmtCond': {'NA': 0, **QUALITY},
    'BsmtExposure': {'NA': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'HeatingQC': dict(QUALITY),
    'KitchenQual': dict(QUALITY),
    'FireplaceQu': {'NA': 0, **QUALITY},
    'GarageType': {'NA': 0, 'CarPort': 1, 'Detchd': 2, 'Attchd': 3, 'Basment': 3, '2Types': 3, 'BuiltIn': 4},
    'GarageFinish': {'NA': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
    'GarageQual': {'NA': 0, **QUALITY},
    'GarageCond': {'NA': 0, **QUALITY},
    'PoolQC': {'NA': 0, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'MasVnrType': {'None': 0, 'BrkCmn': 1, 'BrkFace': 2, 'Stone': 3},
    'Alley': {'NA': 0, 'Grvl': 1, 'Pave': 2},
    'Functional': {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
    'LotShape': {'Reg': 1, 'IR1': 2, 'IR2': 3, 'IR3': 4},
    'LandSlope': {'Gtl': 1, 'Mod': 2, 'Sev': 3},
    'PavedDrive': {'N': 1, 'P': 2, 'Y': 3},
    'Electrical': {'FuseP': 1, 'FuseF': 2, 'FuseA': 3, 'SBrkr': 4, 'Mix': 2},
    'Utilities': {'ELO': 1, 'NoSeWa': 2, 'NoSewr': 3, 'AllPub': 4},
    'OverallQual': {i: i for i in range(1, 11)},
    'OverallCond': {i: i for i in range(1, 11)},
}

NONE_CODES = ('NA', 'None')

BINARY = {
    'CentralAir': {'N': 0, 'Y': 1},
    'Street': {'Grvl': 0, 'Pave': 1},
}

_HOUSE_STYLES = ['1Story', '1.5Fin', '1.5Unf', '2Story', '2.5Fin', '2.5Unf', 'SFoyer', 'SLvl']

_BLDG_TYPES = ['1Fam', '2fmCon', 'Duplex', 'TwnhsE', 'Twnhs']

_LOT_CONFIGS = ['Inside', 'Corner', 'CulDSac', 'FR2', 'FR3']

_CONDITIONS = ['Norm', 'Feedr', 'Artery', 'RRNn', 'RRNe', 'RRAn', 'RRAe', 'PosN', 'PosA']

_FENCES = ['NA', 'GdPrv', 'MnPrv', 'GdWo', 'MnWw']

def _rest(codes, value, **explicit):
    return {c: explicit.get(c, value) for c in codes}


# 뜻이 섞인 열을 속성별 0/1 열로 분해한다. HouseStyle → 층수·반층·상층마감.
DECOMPOSE = {
    'HouseStyle': {
        'HouseStyle_Floors': ('층수', {'1Story': 1, '1.5Fin': 1.5, '1.5Unf': 1.5, '2Story': 2, '2.5Fin': 2.5,
                                      '2.5Unf': 2.5, 'SFoyer': 1.5, 'SLvl': 1.5}),
        'HouseStyle_UpperFin': ('위층 마감', _rest(_HOUSE_STYLES, 1, **{'1.5Unf': 0, '2.5Unf': 0})),
        'HouseStyle_Split': ('스플릿', _rest(_HOUSE_STYLES, 0, SFoyer=1, SLvl=1)),
    },
    'BldgType': {
        'BldgType_Units': ('가구 수', {'1Fam': 1, 'TwnhsE': 1, 'Twnhs': 1, 'Duplex': 2, '2fmCon': 2}),
        'BldgType_SharedWalls': ('타운하우스 공유 벽 수', {'1Fam': 0, '2fmCon': 0, 'Duplex': 0, 'TwnhsE': 1, 'Twnhs': 2}),
        'BldgType_Conversion': ('개조', _rest(_BLDG_TYPES, 0, **{'2fmCon': 1})),
    },
    'LotConfig': {
        'LotConfig_Frontages': ('도로 접면 수', {'Inside': 1, 'CulDSac': 1, 'Corner': 2, 'FR2': 2, 'FR3': 3}),
        'LotConfig_Corner': ('코너', _rest(_LOT_CONFIGS, 0, Corner=1)),
        'LotConfig_CulDSac': ('막다른 길', _rest(_LOT_CONFIGS, 0, CulDSac=1)),
    },
    'Condition': {
        'Condition_Road': ('큰 도로', _rest(_CONDITIONS, 0, Feedr=1, Artery=2)),
        'Condition_Rail': ('철도', _rest(_CONDITIONS, 0, RRNn=1, RRNe=1, RRAn=2, RRAe=2)),
        'Condition_Pos': ('긍정 요소', _rest(_CONDITIONS, 0, PosN=1, PosA=2)),
    },
    'Fence': {
        'Fence_Privacy': ('프라이버시 울타리', _rest(_FENCES, 0, MnPrv=1, GdPrv=2)),
        'Fence_Wood': ('목재 울타리', _rest(_FENCES, 0, MnWw=1, GdWo=2)),
    },
}

DECOMPOSE_SOURCES = {'HouseStyle': ['HouseStyle'], 'BldgType': ['BldgType'], 'LotConfig': ['LotConfig'],
                     'Condition': ['Condition1', 'Condition2'], 'Fence': ['Fence']}


# 외장재 15종을 재료 계열 5개로 묶는다. 비닐이 기준이고 두 면이 다르면 혼합 표시를 켠다.
EXTERIOR_FAMILY = {
    'VinylSd': 'Vinyl',
    'MetalSd': 'Metal',
    'Wd Sdng': 'Wood', 'WdShing': 'Wood', 'Plywood': 'Wood', 'HdBoard': 'Wood',
    'CemntBd': 'CementStucco', 'Stucco': 'CementStucco', 'ImStucc': 'CementStucco',
    'BrkFace': 'BrickStone', 'BrkComm': 'BrickStone', 'Stone': 'BrickStone', 'CBlock': 'BrickStone',
    'AsbShng': 'ShingleOther', 'AsphShn': 'ShingleOther', 'Other': 'ShingleOther', 'PreCast': 'ShingleOther',
}

EXTERIOR_BASE = 'Vinyl'

EXTERIOR_COLUMNS = {
    'Exterior_Metal': ('금속', 'Metal'),
    'Exterior_Wood': ('목재', 'Wood'),
    'Exterior_CementStucco': ('시멘트·스터코', 'CementStucco'),
    'Exterior_BrickStone': ('벽돌·석재', 'BrickStone'),
    'Exterior_ShingleOther': ('슁글·기타', 'ShingleOther'),
}

EXTERIOR_MIXED = ('Exterior_Mixed', '외장재 혼합')

EXTERIOR2_SPELLING = {'CmentBd': 'CemntBd', 'Wd Shng': 'WdShing', 'Brk Cmn': 'BrkComm'}


# These 9 columns have no defensible order. The 0/1 columns made here are dropped and target-encoded instead.
# 이 9개 열은 순서가 없다. 여기서 만든 0/1 열은 버리고 타깃 인코딩으로 대체한다(아래 FACTOR_COLUMNS).
ONEHOT = {
    'MSZoning': [('MSZoning_RM', 'RM', ['RM']), ('MSZoning_RH', 'RH', ['RH']), ('MSZoning_FV', 'FV', ['FV']),
                 ('MSZoning_C', '상업', ['C (all)'])],
    'LandContour': [('LandContour_Bnk', '둔덕', ['Bnk']), ('LandContour_HLS', '언덕 사면', ['HLS']),
                    ('LandContour_Low', '저지대', ['Low'])],
    'RoofStyle': [('RoofStyle_Hip', '모임지붕', ['Hip']),
                  ('RoofStyle_Other', '기타 지붕 형태', ['Flat', 'Gambrel', 'Mansard', 'Shed'])],
    'RoofMatl': [('RoofMatl_Wood', '목재 지붕', ['WdShngl', 'WdShake']),
                 ('RoofMatl_Other', '기타 지붕 재료', ['Tar&Grv', 'Metal', 'Membran', 'Roll', 'ClyTile'])],
    'Foundation': [('Foundation_PConc', '타설 콘크리트', ['PConc']), ('Foundation_CBlock', '콘크리트 블록', ['CBlock']),
                   ('Foundation_BrkTil', '벽돌·타일', ['BrkTil'])],
    'Heating': [('Heating_HotWater', '온수·증기', ['GasW', 'OthW']),
                ('Heating_OldFurnace', '구식 난로', ['Grav', 'Wall', 'Floor'])],
    'MiscFeature': [('MiscFeature_Shed', '창고', ['Shed']),
                    ('MiscFeature_Other', '기타 시설', ['Gar2', 'Othr', 'TenC', 'Elev'])],
    'SaleType': [('SaleType_New', '신축 판매', ['New']), ('SaleType_Contract', '계약 판매', ['ConLD', 'ConLI', 'ConLw', 'Con']),
                 ('SaleType_COD', '법원·유산', ['COD'])],
    'SaleCondition': [('SaleCondition_Partial', '미완공 신축', ['Partial']),
                      ('SaleCondition_Abnorml', '비정상 거래', ['Abnorml']),
                      ('SaleCondition_Special', '특수 거래', ['Family', 'Alloca', 'AdjLand'])],
}

ONEHOT_BASE = {'MSZoning': ['RL'], 'LandContour': ['Lvl'], 'RoofStyle': ['Gable'], 'RoofMatl': ['CompShg'],
               'Foundation': ['Slab', 'Stone', 'Wood'], 'Heating': ['GasA'], 'MiscFeature': ['NA'],
               'SaleType': ['WD', 'CWD', 'VWD', 'Oth'], 'SaleCondition': ['Normal']}


# The only column that uses the price. Per-neighborhood unit price, and the training row's own value is left out.
# 가격을 쓰는 유일한 열. 동네별 평당가이고, train 행에서는 자기 행을 뺀 값을 쓴다.
UNIT_PRICE = 'Neighborhood_UnitPrice'


# MSSubClass는 HouseStyle + BldgType + 건축연도로 98.4%가 복원되므로 지운다.
DROP = ['MSSubClass']

NUMERIC = ['LotFrontage', 'LotArea', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
           'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
           'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars',
           'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
           'MiscVal', 'MoSold', 'YrSold']

CATEGORICAL_FEATURES = (list(ORDINAL) + list(BINARY)
                        + [col for group in DECOMPOSE.values() for col in group]
                        + list(EXTERIOR_COLUMNS) + [EXTERIOR_MIXED[0]]
                        + [col for cols in ONEHOT.values() for col, _, _ in cols]
                        + [UNIT_PRICE])

FEATURES = CATEGORICAL_FEATURES + NUMERIC

FIXED_FEATURES = [f for f in FEATURES if f != UNIT_PRICE]

SOURCE_COLUMNS = (list(ORDINAL) + list(BINARY) + [c for cols in DECOMPOSE_SOURCES.values() for c in cols]
                  + ['Exterior1st', 'Exterior2nd'] + list(ONEHOT) + ['Neighborhood'] + DROP)

def _map(series, mapping, name, numeric=True):
    unknown = set(series.dropna().unique()) - set(mapping)
    if unknown:
        raise ValueError(f'{name}: 매핑에 없는 값 {sorted(map(str, unknown))}')
    mapped = series.map(mapping)
    return mapped.astype(float) if numeric else mapped


# 가격을 쓰지 않는 인코딩 열과 수치형 열을 만든다. Id·Neighborhood는 함께 넘긴다.
def encode_fixed(df):
    cols = {}
    for col, mapping in {**ORDINAL, **BINARY}.items():
        cols[col] = _map(df[col], mapping, col)
    for group, attrs in DECOMPOSE.items():
        sources = DECOMPOSE_SOURCES[group]
        for new, (_, mapping) in attrs.items():
            values = [_map(df[s], mapping, f'{s}→{new}') for s in sources]
            cols[new] = values[0] if len(values) == 1 else np.maximum(*values)
    family1 = _map(df['Exterior1st'], EXTERIOR_FAMILY, 'Exterior1st', numeric=False)
    family2 = _map(df['Exterior2nd'].replace(EXTERIOR2_SPELLING), EXTERIOR_FAMILY, 'Exterior2nd', numeric=False)
    for new, (_, family) in EXTERIOR_COLUMNS.items():
        cols[new] = (family1 == family).astype(float)
    cols[EXTERIOR_MIXED[0]] = (family1 != family2).astype(float)
    for col, groups in ONEHOT.items():
        known = set(ONEHOT_BASE[col]) | {v for _, _, members in groups for v in members}
        unknown = set(df[col].dropna().unique()) - known
        if unknown:
            raise ValueError(f'{col}: 매핑에 없는 값 {sorted(unknown)}')
        for new, _, members in groups:
            cols[new] = df[col].isin(members).astype(float).where(df[col].notna())
    for col in NUMERIC:
        cols[col] = df[col].astype(float)
    keep = [c for c in ('Id', 'Neighborhood', 'SalePrice') if c in df.columns]
    out = pd.concat([df[keep], pd.DataFrame(cols, index=df.index)], axis=1)
    return out[keep + FIXED_FEATURES]

### `Utilities` is dropped / `Utilities`는 뺀다

뺀 이유는 0절에 적었다.<br>
인코딩은 그대로 하고 설계행렬을 만들 때만 뺀다(`build(drop_cols=...)`) — 터미널이 90개에서 89개가 된다.

## Folds — leakage control and the design matrix / fold별 누출 통제와 설계행렬

### Leakage control / 누출 통제

In [11]:
# Stage 4 — leakage control. Every answer-touching statistic is fitted on the fold's train rows only.
# 4단계 — 누출 통제. 정답(가격)을 쓰는 통계는 그 fold의 train 행만으로 계산한다.
#   FoldPreprocessor.fit(train)      : 평당가·스케일·타깃 통계를 train 행에서 구한다
#   FoldPreprocessor.transform(other): valid·test 행은 가격을 **읽지 않는다**
#   make_folds                       : fold 배정. 이 노트북은 공유된 fold 표를 쓰고, 그 표가 이 함수의 결과와 같은지 확인한다

SEED = 42

N_FOLDS = 5


# 타깃 인코딩의 평활 강도. 행이 적은 수준일수록 전체 평균 쪽으로 당겨진다.
SMOOTHING_M = 10


# 0에 뜻이 없고 값이 0에서 멀리 떨어진 열만 z-score한다. 나머지는 표준편차로만 나눈다.
ZSCORE = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'YrSold', 'MoSold', 'LotFrontage']

def scale_method(feature):
    return 'zscore' if feature in ZSCORE else 'divide'


# fit() sees the fold's training rows only; transform() never reads a price. This is the leakage boundary.
# fit()은 그 fold의 train 행만 본다. transform()은 가격을 읽지 않는다. 누출을 막는 경계가 바로 여기다.
class FoldPreprocessor:

    def fit(self, enc):
        price = enc['SalePrice'].to_numpy(float)
        unit = price / enc['GrLivArea'].to_numpy(float)
        groups = pd.Series(unit).groupby(enc['Neighborhood'].to_numpy())
        self.unit_mean_ = float(unit.mean())
        self.unit_sum_ = groups.sum().to_dict()
        self.unit_count_ = groups.count().to_dict()
        S, n = self._neighborhood_stats(enc)
        m, mu = SMOOTHING_M, self.unit_mean_
        loo = (S - unit + m * mu) / (n - 1 + m)
        raw = self._assemble(enc, loo)
        self.mean_ = raw.mean(axis=0)
        std = raw.std(axis=0)
        self.std_ = np.where(std > 0, std, 1.0)
        self.method_ = np.array([scale_method(f) for f in FEATURES])
        self.center_ = np.where(self.method_ == 'zscore', self.mean_, 0.0)
        log_price = np.log(price)
        self.target_mean_ = float(log_price.mean())
        self.target_std_ = float(log_price.std())
        self.n_train_ = len(enc)
        self.X_train_ = (raw - self.center_) / self.std_
        return self

    def transform(self, enc):
        S, n = self._neighborhood_stats(enc)
        m, mu = SMOOTHING_M, self.unit_mean_
        return (self._assemble(enc, (S + m * mu) / (n + m)) - self.center_) / self.std_

    def target(self, sale_price):
        return (np.log(np.asarray(sale_price, float)) - self.target_mean_) / self.target_std_

    def to_log_price(self, z):
        return np.asarray(z) * self.target_std_ + self.target_mean_

    def scale_table(self):
        return pd.DataFrame({'feature': FEATURES, 'method': self.method_, 'mean': self.mean_, 'std': self.std_})

    def _neighborhood_stats(self, enc):
        nb = enc['Neighborhood']
        S = nb.map(self.unit_sum_).fillna(0.0).to_numpy(float)
        n = nb.map(self.unit_count_).fillna(0).to_numpy(float)
        return S, n

    @staticmethod
    def _assemble(enc, unit_price):
        raw = pd.DataFrame(enc[FIXED_FEATURES].to_numpy(float), columns=FIXED_FEATURES)
        raw[UNIT_PRICE] = unit_price
        raw = raw[FEATURES].to_numpy(float)
        if np.isnan(raw).any():
            bad = sorted({FEATURES[j] for j in np.where(np.isnan(raw))[1]})
            raise ValueError(f'재료에 빈 칸이 있다: {bad}')
        return raw


# fold 배정. 이 노트북은 공유된 fold 표를 쓰고, 그 표가 이 함수의 결과와 같은지 확인한다.
def make_folds(n_rows, n_folds=N_FOLDS, seed=SEED):
    fold = np.empty(n_rows, dtype=int)
    for k, (_, valid_idx) in enumerate(KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(np.arange(n_rows))):
        fold[valid_idx] = k
    return fold

@dataclass
class FoldData:
    fold: int
    X_train: np.ndarray
    y_train: np.ndarray
    X_valid: np.ndarray
    y_valid: np.ndarray
    target_mean: float
    target_std: float
    ids_train: np.ndarray
    ids_valid: np.ndarray


# fold k를 valid로 두고 나머지로 전처리를 적합한다.
def fold_data(enc, folds, k):
    train, valid = enc[folds != k].reset_index(drop=True), enc[folds == k].reset_index(drop=True)
    pre = FoldPreprocessor().fit(train)
    return FoldData(k, pre.X_train_, np.log(train['SalePrice'].to_numpy(float)), pre.transform(valid),
                    np.log(valid['SalePrice'].to_numpy(float)), pre.target_mean_, pre.target_std_,
                    train['Id'].to_numpy(), valid['Id'].to_numpy()), pre


# train 행 전체로 적합한다(fold를 나누지 않는 최종 실행용).
def full_data(enc):
    pre = FoldPreprocessor().fit(enc)
    return FoldData(-1, pre.X_train_, np.log(enc['SalePrice'].to_numpy(float)), None, None,
                    pre.target_mean_, pre.target_std_, enc['Id'].to_numpy(), None), pre

### Two outliers stay on the train side / 이상치 두 채는 train 쪽에 둔다

이렇게 다루는 이유는 0절에 적었다. `build`가 두 가지를 한다.

- `pin_ids` : 두 채의 fold 번호를 −1로 바꿔 어느 fold에서도 valid 행이 되지 않게 한다. 다른 행의 fold 번호와 OOF용 내부 fold 번호는 그대로다. 이 분할에서는 fold 1 · 2의 valid 행이 291행에서 290행이 된다.
- `stat_exclude_ids` : 타깃 인코딩의 수준별 평균과 동네 평균 단가를 낼 때 두 채를 뺀다. 두 채 자신의 값은 나머지 행의 통계로 채운다.

### Out-of-fold target encoding / OOF 타깃 인코딩

타깃 인코딩은 범주의 각 수준을 그 수준에 속한 행들의 평균 정답으로 바꾼다.<br>
그대로 쓰면 각 행의 정답이 그 행의 피쳐에 섞여 들어간다.

LOO(자기 행만 빼고 평균)로 고치면 오히려 나빠진다. 수준의 정답 합을 $S_l$, 행 수를 $n_l$이라 하면

$$\text{LOO}_i = \frac{S_l - y_i + m\mu}{n_l - 1 + m} = -\frac{y_i}{n_l - 1 + m} + \text{const}_l$$

이 값은 그 행 정답의 완전한 아핀(일차) 함수다 — 수준만 알아내고 상수항을 걷어 내면 $y_i$가 그대로 나온다.<br>
실제로 재 보면 상관계수가 −1.0이다.<br>
이 열을 GP에 주면 train 오차는 0 가까이 떨어지는데 valid는 0.4를 넘고, GCV는 끝까지 알아채지 못한다 — 누출이 모델을 맞추는 단계가 아니라, 그 전에 만든 입력 열에 있기 때문이다.

OOF는 train 행을 다시 내부 5 fold로 나누고, 각 행에 나머지 네 fold로 낸 통계를 준다.<br>
자기 정답이 자기 피쳐에 들어가지 않으니 거꾸로 계산해 낼 것도 없다.

수준마다 평균 하나를 모든 행에 똑같이 주는 방식과도 다르다.<br>
그 방식에서는 자기 행이 자기 수준 평균에 $1/(n_l+m)$만큼 섞여, train 쪽 성능이 실제보다 좋아 보인다.<br>
OOF에서는 train 행도 '자기가 빠진 평균'을 보므로 valid 행과 같은 조건이 된다.

In [12]:
# Stage 5 — terminals and the design matrix. 78 engineered + 11 out-of-fold target encodings = 89.
# 5단계 — 터미널과 설계행렬. 가공 78열 + OOF 타깃 인코딩 11열 = 89개.
#   FACTOR_COLUMNS   : 순서가 없어 타깃 인코딩으로 보내는 11열
#   KEPT             : 그 11열의 인코딩을 뺀 나머지 79열. build가 drop_cols(Utilities)를 빼서 78열이 된다
#   _Pre             : FoldPreprocessor와 같되, 동네 평균 단가 통계에서 이상치 두 채를 뺀다
#   target_encode_oof: OOF 타깃 인코딩. 위에서 설명한 방식을 구현한 함수다
#   build            : fold 하나의 Design(train·valid·test 행렬과 Id)을 만든다. 이상치 두 채는 pin_ids로 train 쪽에 둔다

OOF_FOLDS = 5

TE_ENCODINGS = ('target', 'loo', 'oof', 'mixed')

FACTOR_ONEHOT = list(ONEHOT)

FACTOR_ORDINAL = ['GarageType', 'MasVnrType']


# The 11 nominal columns with no defensible order. These get the out-of-fold target encoding.
# 순서가 없는 명목형 11열. 이 열들은 OOF 타깃 인코딩으로 바꾼다.
FACTOR_COLUMNS = FACTOR_ONEHOT + FACTOR_ORDINAL

DROPPED = [new for col in FACTOR_ONEHOT for new, _, _ in ONEHOT[col]] + FACTOR_ORDINAL


# 재료 79열. 위 11열의 인코딩을 뺀 나머지다. build가 drop_cols(Utilities)를 빼서 터미널은 78 + 11 = 89개다.
KEPT = [f for f in FEATURES if f not in set(DROPPED)]


# 컬럼의 수준 목록. 드문 값 병합은 원래 인코딩 규칙을 그대로 따른다.
def factor_levels(col):
    if col in ONEHOT:
        groups = [list(ONEHOT_BASE[col])] + [list(vals) for _, _, vals in ONEHOT[col]]
        return groups
    mapping = ORDINAL[col]
    codes = sorted(set(mapping.values()))
    return [[k for k, v in mapping.items() if v == code] for code in codes]

def level_index(col):
    return {v: i for i, group in enumerate(factor_levels(col)) for v in group}


# theta 초기값. 이 노트북에서는 쓰지 않는다.
def theta_base(col, std):
    if col in ONEHOT:
        n = len(factor_levels(col))
        return (np.arange(n, dtype=float) - (n - 1) / 2) / max(n - 1, 1) * 2
    codes = np.array(sorted(set(ORDINAL[col].values())), dtype=float)
    return codes / (std if std > 0 else 1.0)


# The LOO variant is kept only to show why it is unusable: it is an exact affine function of the row's own answer.
# LOO 변형은 '왜 쓸 수 없는지'를 보이기 위해서만 남겼다. 그 행 정답의 완전한 아핀 함수다(상관 -1.0).
# 수준 평균을 모든 행에 똑같이 주는 방식은 자기 행이 자기 수준 평균에 1/(n+m)만큼 섞여, train 쪽 점수가 실제보다 좋게 나온다.
def target_encode(codes_tr, y_tr, n_lv, codes_other=None, m=SMOOTHING_M, loo=False, exclude=None):
    keep = np.ones(len(y_tr), bool) if exclude is None else ~np.asarray(exclude, bool)
    mu = float(y_tr[keep].mean())
    S = np.bincount(codes_tr[keep], weights=y_tr[keep], minlength=n_lv)
    n = np.bincount(codes_tr[keep], minlength=n_lv).astype(float)
    lvl = (S + m * mu) / (n + m)
    own = keep.astype(float)
    tr = ((S[codes_tr] - own * y_tr + m * mu) / (n[codes_tr] - own + m)) if loo else lvl[codes_tr]
    return tr, (lvl[codes_other] if codes_other is not None else None), lvl


# Out-of-fold: a training row sees the statistic of the four inner folds it does not belong to.
# OOF. train 행은 자기가 속하지 않은 내부 네 fold의 통계를 본다.
# 자기 정답이 자기 피쳐에 들어가지 않으므로 역산할 대상이 없다. valid·test 행은 train 전체 통계를 쓴다.
# exclude가 True인 행(이상치 두 채)은 어느 통계에도 넣지 않는다. 그 행도 값은 받는다.
def target_encode_oof(codes_tr, y_tr, n_lv, inner, codes_other=None, m=SMOOTHING_M, exclude=None):
    keep = np.ones(len(y_tr), bool) if exclude is None else ~np.asarray(exclude, bool)
    tr = np.empty(len(codes_tr), float)
    for j in np.unique(inner):
        ref = (inner != j) & keep
        y_ref, c_ref = y_tr[ref], codes_tr[ref]
        mu_j = float(y_ref.mean())
        S_j = np.bincount(c_ref, weights=y_ref, minlength=n_lv)
        n_j = np.bincount(c_ref, minlength=n_lv).astype(float)
        lvl_j = (S_j + m * mu_j) / (n_j + m)
        tr[inner == j] = lvl_j[codes_tr[inner == j]]
    mu = float(y_tr[keep].mean())
    S = np.bincount(codes_tr[keep], weights=y_tr[keep], minlength=n_lv)
    n = np.bincount(codes_tr[keep], minlength=n_lv).astype(float)
    lvl = (S + m * mu) / (n + m)
    return tr, (lvl[codes_other] if codes_other is not None else None), lvl


# Same as FoldPreprocessor, except that the two outliers are left out of the neighbourhood unit-price statistics.
# FoldPreprocessor와 같고, 동네 평균 단가 통계에서만 이상치 두 채를 뺀다. 두 채 자신은 valid 행처럼 값을 받는다.
class _Pre(FoldPreprocessor):

    def __init__(self, exclude_ids):
        self.exclude_ids = tuple(exclude_ids)

    def fit(self, enc):
        price = enc['SalePrice'].to_numpy(float)
        unit = price / enc['GrLivArea'].to_numpy(float)
        ex = enc['Id'].isin(self.exclude_ids).to_numpy()
        nb = enc['Neighborhood'].to_numpy()
        groups = pd.Series(unit[~ex]).groupby(nb[~ex])
        self.unit_mean_ = float(unit[~ex].mean())
        self.unit_sum_ = groups.sum().to_dict()
        self.unit_count_ = groups.count().to_dict()
        S, n = self._neighborhood_stats(enc)
        m, mu = SMOOTHING_M, self.unit_mean_
        loo = np.where(ex, (S + m * mu) / (n + m), (S - unit + m * mu) / (n - 1 + m))
        raw = self._assemble(enc, loo)
        self.mean_ = raw.mean(axis=0)
        std = raw.std(axis=0)
        self.std_ = np.where(std > 0, std, 1.0)
        self.method_ = np.array([scale_method(f) for f in FEATURES])
        self.center_ = np.where(self.method_ == 'zscore', self.mean_, 0.0)
        log_price = np.log(price)
        self.target_mean_ = float(log_price.mean())
        self.target_std_ = float(log_price.std())
        self.n_train_ = len(enc)
        self.X_train_ = (raw - self.center_) / self.std_
        return self


# fold_data와 같고, 통계에서 뺄 행이 있으면 _Pre로 적합한다.
def _fold_data(enc, folds, k, exclude_ids):
    if not exclude_ids:
        return fold_data(enc, folds, k)
    train, valid = enc[folds != k].reset_index(drop=True), enc[folds == k].reset_index(drop=True)
    pre = _Pre(exclude_ids).fit(train)
    return FoldData(k, pre.X_train_, np.log(train['SalePrice'].to_numpy(float)), pre.transform(valid),
                         np.log(valid['SalePrice'].to_numpy(float)), pre.target_mean_, pre.target_std_,
                         train['Id'].to_numpy(), valid['Id'].to_numpy()), pre


# full_data와 같고, 통계에서 뺄 행이 있으면 _Pre로 적합한다.
def _full_data(enc, exclude_ids):
    if not exclude_ids:
        return full_data(enc)
    pre = _Pre(exclude_ids).fit(enc)
    return FoldData(-1, pre.X_train_, np.log(enc['SalePrice'].to_numpy(float)), None, None,
                         pre.target_mean_, pre.target_std_, enc['Id'].to_numpy(), None), pre


# train 행 1,453 전부를 fold 분할에 쓴다(따로 떼어 두는 행이 없다).
def _all_rows(p):
    from types import SimpleNamespace
    return SimpleNamespace(**{**vars(p), 'train_splitted': p.train, 'holdout': p.train.iloc[:0]})


# Pinning moves two rows into a fold's training side; every other row keeps its inner fold number.
# 고정 때문에 train 쪽으로 새로 들어온 행만 내부 fold Id % 5번에 넣는다. 나머지 행의 내부 fold 번호는 그대로다.
def _inner_folds(n_train, new_rows, ids_train):
    if not new_rows.any():
        return make_folds(n_train, n_folds=OOF_FOLDS)
    inner = np.empty(n_train, int)
    inner[~new_rows] = make_folds(int((~new_rows).sum()), n_folds=OOF_FOLDS)
    inner[new_rows] = np.asarray(ids_train)[new_rows] % OOF_FOLDS
    return inner


# fold 하나의 재료. train·valid·test 행렬과 Id, 그리고 터미널 이름이 들어 있다.
@dataclass
class Design:
    fold: int
    X_train: np.ndarray
    y_train: np.ndarray
    codes_train: dict
    X_valid: np.ndarray
    y_valid: np.ndarray
    codes_valid: dict
    theta_init: dict
    n_levels: dict
    features: list
    ids_train: np.ndarray
    ids_valid: np.ndarray
    encoding: str = 'theta'
    arg_names: list = None
    n_cols: int = None
    loo_levels: dict = None
    X_holdout: np.ndarray = None
    y_holdout: np.ndarray = None
    codes_holdout: dict = None
    ids_holdout: np.ndarray = None
    X_test: np.ndarray = None
    codes_test: dict = None
    ids_test: np.ndarray = None

def _codes(raw, cols):
    out = {}
    for col in cols:
        idx = level_index(col)
        src = raw['Condition1'] if col == 'Condition' else raw[col]
        out[col] = src.map(idx).to_numpy(np.int64)
        assert not np.any(out[col] < 0), col
    return out


# One fold's design matrix: 78 engineered columns + 11 out-of-fold target encodings = 89 terminals.
# fold 하나의 설계행렬. 가공 78열 + OOF 타깃 인코딩 11열 = 터미널 89개.
#   drop_cols        : 터미널에서 뺄 열(Utilities). 인코딩은 그대로 하고 설계행렬에서만 뺀다
#   pin_ids          : 모든 fold에서 train 쪽에 둘 Id. fold 번호를 -1로 두어 어느 fold의 valid에도 들어가지 않는다
#   stat_exclude_ids : 가격에서 나온 통계(타깃 인코딩 수준 평균, 동네 평균 단가)에서 뺄 Id
#   rows='all'       : train 행 1,453 전부를 fold 분할에 쓴다. 따로 떼어 두는 행(holdout)이 없다
def build(fold=None, folds=None, prepared=None, encoding='theta', holdout=False, test=False,
          inner_seed=None, drop_ids=(), pin_ids=(), stat_exclude_ids=(), rows='splitted', drop_cols=()):
    unknown = set(drop_cols) - set(KEPT)
    if unknown:
        raise ValueError(f'drop_cols에 재료에 없는 열이 있다: {sorted(unknown)}')
    kept = [f for f in KEPT if f not in set(drop_cols)]
    p = prepared or prepare()
    if rows == 'all':
        p = _all_rows(p)
    elif rows != 'splitted':
        raise ValueError(f"rows는 'all' 또는 'splitted'다: {rows!r}")
    ts = p.train_splitted
    enc_all = encode_fixed(ts)
    keep = [FEATURES.index(f) for f in kept]
    if drop_ids:
        m = ~ts['Id'].isin(list(drop_ids)).to_numpy()
        if folds is None and fold is not None:
            folds = make_folds(len(enc_all))
        if folds is not None:
            folds = np.asarray(folds)[m]
        ts, enc_all = ts[m].reset_index(drop=True), enc_all[m].reset_index(drop=True)
    if fold is None and inner_seed is None:
        fd, pre = _full_data(enc_all, stat_exclude_ids)
        tr_raw, va_raw = ts, None
        new_rows = np.zeros(len(fd.y_train), bool)
    else:
        if fold is None:
            f, k = make_folds(len(enc_all), seed=inner_seed), 0
        else:
            f = folds if folds is not None else make_folds(len(enc_all))
            k = fold
        f0 = np.asarray(f)
        f = f0
        if pin_ids:
            f = f0.copy()
            f[ts['Id'].isin(list(pin_ids)).to_numpy()] = -1
        fd, pre = _fold_data(enc_all, f, k, stat_exclude_ids)
        tr_raw = ts[f != k].reset_index(drop=True)
        va_raw = ts[f == k].reset_index(drop=True)
        new_rows = f0[f != k] == k
    std = {c: float(pre.std_[FEATURES.index(c)]) for c in FACTOR_ORDINAL}
    Xtr = np.ascontiguousarray(fd.X_train[:, keep])
    Xva = None if fd.X_valid is None else np.ascontiguousarray(fd.X_valid[:, keep])
    codes_tr = _codes(tr_raw, FACTOR_COLUMNS)
    codes_va = None if va_raw is None else _codes(va_raw, FACTOR_COLUMNS)
    n_lv = {c: len(factor_levels(c)) for c in FACTOR_COLUMNS}
    names = list(kept)
    loo_lvl = None

    if encoding in TE_ENCODINGS:
        inner = (_inner_folds(len(fd.y_train), new_rows, fd.ids_train) if encoding == 'oof' else None)
        excl = np.isin(fd.ids_train, list(stat_exclude_ids)) if stat_exclude_ids else None
        extra_tr, extra_va, loo_lvl = [], [], {}
        for c in FACTOR_COLUMNS:
            other = None if codes_va is None else codes_va[c]
            if encoding == 'oof':
                t, v, lvl = target_encode_oof(codes_tr[c], fd.y_train, n_lv[c], inner, other, exclude=excl)
            else:
                t, v, lvl = target_encode(codes_tr[c], fd.y_train, n_lv[c], other,
                                          loo=(encoding == 'loo'), exclude=excl)
            mu, sd = float(t.mean()), float(t.std())
            sd = sd if sd > 1e-12 else 1.0
            extra_tr.append((t - mu) / sd)
            if v is not None:
                extra_va.append((v - mu) / sd)
            loo_lvl[c] = dict(levels=(lvl - mu) / sd, mean=mu, std=sd, raw_levels=lvl)
            names.append(f'{c}_TE')
        Xtr = np.ascontiguousarray(np.column_stack([Xtr] + extra_tr))
        if Xva is not None:
            Xva = np.ascontiguousarray(np.column_stack([Xva] + extra_va))
        if encoding == 'mixed':
            arg_names = list(names) + [f'CAT_{c}' for c in FACTOR_COLUMNS]
            n_cols = len(names)
        else:
            codes_tr, codes_va, n_lv = {}, ({} if codes_va is not None else None), {}
            arg_names, n_cols = list(names), len(names)
    elif encoding == 'numeric':
        codes_tr, codes_va, n_lv = {}, ({} if codes_va is not None else None), {}
        arg_names, n_cols = list(kept), len(kept)
    else:
        arg_names = list(kept) + [f'FAC_{c}' for c in FACTOR_COLUMNS]
        n_cols = len(kept)

    def _block(frame):
        src = frame if 'SalePrice' in frame.columns else frame.assign(SalePrice=1.0)
        X = np.ascontiguousarray(pre.transform(encode_fixed(src))[:, keep])
        cd = _codes(frame, FACTOR_COLUMNS)
        if encoding in TE_ENCODINGS:
            X = np.ascontiguousarray(np.column_stack(
                [X] + [loo_lvl[c]['levels'][cd[c]] for c in FACTOR_COLUMNS]))
            if encoding != 'mixed':
                cd = {}
        return X, cd

    Xho = yho = codes_ho = ids_ho = None
    if holdout and len(p.holdout):
        Xho, codes_ho = _block(p.holdout)
        yho = np.log(p.holdout['SalePrice'].to_numpy(float))
        ids_ho = p.holdout['Id'].to_numpy()
    Xte = codes_te = ids_te = None
    if test:
        Xte, codes_te = _block(p.test)
        ids_te = p.test['Id'].to_numpy()


    return Design(
        fold=-1 if fold is None else fold,
        X_train=Xtr, y_train=fd.y_train, codes_train=codes_tr,
        X_valid=Xva, y_valid=fd.y_valid, codes_valid=codes_va,
        theta_init=({} if encoding in TE_ENCODINGS or encoding == 'numeric' else
                    {c: theta_base(c, std.get(c, 1.0)) for c in FACTOR_COLUMNS}),
        n_levels=n_lv, features=names, ids_train=fd.ids_train, ids_valid=fd.ids_valid,
        encoding=encoding, arg_names=arg_names, n_cols=n_cols, loo_levels=loo_lvl,
        X_holdout=Xho, y_holdout=yho, codes_holdout=codes_ho, ids_holdout=ids_ho,
        X_test=Xte, codes_test=codes_te, ids_test=ids_te)

### Build the design matrices / 설계행렬 만들기

In [13]:
# Every usable training row goes into the five folds. Nothing is held out.
# 쓸 수 있는 train 행을 모두 fold 다섯 개에 나눠 넣는다. 따로 떼어 두는 행은 없다.
train_fixed, test_fixed, CHANGES = fix_contradictions(train_csv, test_csv)
train_fixed, test_fixed, FILLS = fill_missing(train_fixed, test_fixed)

PREP = SimpleNamespace(train=train_fixed, train_splitted=train_fixed,
                       holdout=train_fixed.iloc[:0], test=test_fixed)
FOLDS = train_fixed.Id.map(FOLD_OF).to_numpy()
assert not pd.isna(FOLDS).any(), 'a training row has no fold number!!'
FOLDS = FOLDS.astype(int)

DES = {f: build(fold=f, folds=FOLDS, prepared=PREP, encoding='oof', test=True, rows='all',
                drop_cols=DROP_COLS, pin_ids=PIN_IDS, stat_exclude_ids=STAT_EXCLUDE_IDS) for f in range(5)}
print(f'rows fixed {len(train_fixed):,} · value changes {len(CHANGES)} · fills {len(FILLS)}')
for f in range(5):
    d = DES[f]
    print(f'  fold {f}: train {len(d.y_train):,} · valid {len(d.y_valid):,} '
          f'· terminals {d.X_train.shape[1]} · test {len(d.ids_test):,}')
print(f'max RSS {RSS_MB():.0f} MB')

rows fixed 1,453 · value changes 6 · fills 13933
  fold 0: train 1,162 · valid 291 · terminals 89 · test 1,459
  fold 1: train 1,163 · valid 290 · terminals 89 · test 1,459
  fold 2: train 1,163 · valid 290 · terminals 89 · test 1,459
  fold 3: train 1,163 · valid 290 · terminals 89 · test 1,459
  fold 4: train 1,163 · valid 290 · terminals 89 · test 1,459
max RSS 262 MB


### Preprocessing sanity check / 전처리 정합성 검사

In [14]:
# Check 1 of 4 — the data. Every assert here is against the data itself, not a number I measured earlier.
# 검증 1/4 — **데이터**. 여기 있는 assert는 모두 바깥 기록이 아니라 데이터 자체를 근거로 한다.
_dropped = sorted(int(i) for i in set(train_csv.Id) - set(train_fixed.Id))
assert len(_dropped) == 7, f'{len(_dropped)} rows were dropped, expected 7: {_dropped}!!'
assert set(train_fixed.Id) == set(fold_tbl.Id), 'the fixed training rows and the fold table differ!!'

# The shared fold table has to be what make_folds produces here — this is where a sklearn version
# difference would show up, and it would change every fold silently.
# 공유된 fold 표는 이 환경에서 make_folds를 돌린 결과와 같아야 한다. sklearn 버전 차이가 있으면 여기서 드러나고,
# 놓치면 모든 fold가 소리 없이 달라진다.
assert np.array_equal(make_folds(len(train_fixed)), FOLDS), \
    'make_folds disagrees with the shared fold table (sklearn version drift?)!!'

print(f'contradiction rows removed {_dropped}')
print(f'fold sizes {np.bincount(FOLDS).tolist()} · make_folds agrees with the shared table')

print(f'\n{"fold":>5}{"train":>7}{"valid":>7}{"test":>7}{"terminals":>11}{"y sum":>13}')
for f in range(5):
    d = DES[f]
    assert d.X_train.shape[1] == 89, f'fold {f} has {d.X_train.shape[1]} terminals, not 89!!'
    assert not set(DROP_COLS) & set(d.arg_names), f'fold {f} still has {DROP_COLS} as a terminal!!'
    assert len(d.y_train) + len(d.y_valid) == len(train_fixed), f'fold {f} rows do not add up!!'
    # The two outliers are training rows in every fold, never validation rows.
    # 이상치 두 채는 모든 fold에서 train 행이고, 어느 fold에서도 valid 행이 아니다.
    assert set(PIN_IDS) <= set(d.ids_train) and not set(PIN_IDS) & set(d.ids_valid), \
        f'fold {f}: {PIN_IDS} are not pinned to the train side!!'
    assert np.isfinite(d.X_train).all() and np.isfinite(d.X_valid).all(), f'fold {f} non-finite!!'
    assert np.isfinite(d.X_test).all(), f'fold {f} test matrix has non-finite values!!'
    assert not (set(d.ids_train) & set(d.ids_valid)), f'fold {f} train and valid share a row!!'
    print(f'{f:>5}{len(d.y_train):>7,}{len(d.y_valid):>7,}{len(d.ids_test):>7,}'
          f'{d.X_train.shape[1]:>11}{float(d.y_train.sum()):>13.4f}')

print('\n*** Preprocessing check passed — rows, folds, pins, terminals and finiteness all hold ***')

contradiction rows removed [625, 689, 774, 1231, 1242, 1301, 1335]
fold sizes [291, 291, 291, 290, 290] · make_folds agrees with the shared table

 fold  train  valid   test  terminals        y sum
    0  1,162    291  1,459         89   13981.3067
    1  1,163    290  1,459         89   13991.0307
    2  1,163    290  1,459         89   13976.5664
    3  1,163    290  1,459         89   13976.0928
    4  1,163    290  1,459         89   13978.8537

*** Preprocessing check passed — rows, folds, pins, terminals and finiteness all hold ***


# 4. Define the model / 모델 정의

### **모델은 프로그램들의 개체군이다**
개체 하나는 DEAP 수식 나무 1\~45개를 담은 가변 길이 목록이다.<br>
나무는 터미널 89개와 보호 연산자 8개(`add sub mul div log sqrt max min`)로 만든다 — 보호 연산자는 0으로 나누거나 음수에 로그를 씌워도 오류 없이 유한한 값을 돌려준다.

개체는 이렇게 평가한다 — 나무마다 train 행 전체의 값을 계산해 열 하나를 만들고, 그 열들을 쌓은 행렬로 ARD를 푼다.<br>
이때 나온 PRESS가 적합도의 오차 항이고, 여기에 가격 두 가지를 더한다.

- **노드 가격** : 노드 하나에 1e-5, 고정이다. 아무 일도 하지 않는 노드가 공짜로 붙어 있지 못하게 한다.
- **유전자 가격** : 살아 있는 유전자 하나마다 매긴다. 개체군이 다 자랄 때까지는 0이고, 그 뒤 250세대에 걸쳐 0.003까지 오른다.

## The material — primitives and the evaluator / 재료 — 연산자와 평가기

### Primitives and the expression evaluator / 연산자와 수식 평가기

In [15]:
# Protected operators, the typed primitive set, and the column-wise expression evaluator.
# 보호 연산자, 프리미티브 집합(pset), 그리고 수식을 열 단위로 계산하는 평가기.
#   protected_div/log/sqrt : 0 나누기·음수 로그에서 예외 대신 유한값을 돌려준다
#   make_pset              : 터미널 89개 + 연산자 8개 + ephemeral 상수
#   Evaluator.forward      : 나무 하나를 후위 순회로 한 번에 계산한다(세대마다 수만 번 넘게 불린다)
#   tournament             : 토너먼트 선택(크기 15). 상위 20%가 부모 자리의 96.5%를 차지한다

PROTECT_EPS = 0.001


# Protected operators return a finite value instead of raising, so a candidate never dies of a zero divide.
# 보호 연산자는 오류를 내는 대신 유한한 값을 돌려준다. 그래서 0으로 나누다가 개체가 죽는 일이 없다.
# |분모| <= 0.001이면 나눗셈은 1, |x| <= 0.001이면 log는 0, sqrt는 |x|를 쓴다.
def protected_div(a, b):
    if isinstance(b, np.ndarray):
        out = np.ones_like(b)
        np.divide(a, b, out=out, where=np.abs(b) > PROTECT_EPS)
        return out
    if abs(b) > PROTECT_EPS:
        return a / b
    return np.ones_like(a) if isinstance(a, np.ndarray) else np.float64(1.0)

def protected_log(a):
    if isinstance(a, np.ndarray):
        absa = np.abs(a)
        out = np.zeros_like(absa)
        np.log(absa, out=out, where=absa > PROTECT_EPS)
        return out
    absa = abs(a)
    return np.log(absa) if absa > PROTECT_EPS else np.float64(0.0)

def protected_sqrt(a):
    return np.sqrt(np.abs(a))


# 나무의 높이. 상한(6)을 넘는 교차·돌연변이 결과는 버린다.
def tree_height(tree):
    stack = [0]
    pop, push = stack.pop, stack.append
    best = 0
    for node in tree:
        depth = pop()
        if depth > best:
            best = depth
        if node.__class__ is gp.Primitive:
            depth += 1
            arity = node.arity
            if arity == 2:
                push(depth)
                push(depth)
            elif arity == 1:
                push(depth)
            else:
                stack.extend([depth] * arity)
    return best


# Tournament of 15: draw 15 at random, the lowest fitness wins. It supplies the parents of everything but the elite.
# 토너먼트 15. 무작위로 15개를 뽑아 적합도가 가장 낮은 개체를 고른다. 엘리트를 뺀 나머지 자리의 부모는 모두 이렇게 뽑는다.
# 적합도가 같으면 짧은 개체가 이긴다. 복원 추출이라 순위가 낮은 개체도 뽑힐 수 있다.
def tournament(fitness, length, n, size, rng, entrants=False):
    idx = rng.integers(0, len(fitness), size=(n, size))
    f = fitness[idx]
    tie = np.where(f == f.min(axis=1, keepdims=True), length[idx], np.iinfo(np.int64).max)
    slot = tie.argmin(axis=1)
    winners = idx[np.arange(n), slot]
    return (winners, idx, slot) if entrants else winners

PRIMITIVES = {'add': operator.add, 'sub': operator.sub,
              'mul': operator.mul, 'div': protected_div,
              'log': protected_log, 'sqrt': protected_sqrt,
              'max': np.maximum, 'min': np.minimum}
PRIMITIVES = {k: (v, 1 if k in ('log', 'sqrt') else 2)
              for k, v in PRIMITIVES.items()}

EPHEMERAL = 'c3'

EPHEMERAL_T = 'c4'

N_KEPT = len(KEPT)

ARG_NAMES = list(KEPT) + [f'FAC_{c}' for c in FACTOR_COLUMNS]

K_COL, K_FAC, K_CONST, K_PRIM, K_CAT = 0, 1, 2, 3, 4

class Cat:
    pass

def random_constant():
    return random.uniform(-1.0, 1.0)


# 89 terminals, 8 operators, and an ephemeral constant drawn from U(-1, 1) each time a node is created.
# 터미널 89개, 연산자 8개, 그리고 노드를 만들 때마다 U(-1, 1)에서 뽑는 ephemeral 상수.
def make_pset(names=None):
    names = list(names) if names else ARG_NAMES
    cat = {i for i, n in enumerate(names) if n.startswith('CAT_')}
    if not cat:
        pset = gp.PrimitiveSet('V3', len(names))
        for name, (fn, arity) in PRIMITIVES.items():
            pset.addPrimitive(fn, arity, name=name)
        pset.addEphemeralConstant(EPHEMERAL, random_constant)
    else:
        pset = gp.PrimitiveSetTyped(
            'V4', [Cat if i in cat else float for i in range(len(names))], float)
        for name, (fn, arity) in PRIMITIVES.items():
            pset.addPrimitive(fn, [float] * arity, float, name=name)
        pset.addEphemeralConstant(EPHEMERAL_T, random_constant, float)
    pset.renameArguments(**{f'ARG{i}': n for i, n in enumerate(names)})
    return pset


# 노드 이름 → (종류, 값). 터미널 이름은 renameArguments 뒤에도 ARGi로 남으므로 value를 쓴다.
def node_table(pset, n_cols=None):
    n_cols = N_KEPT if n_cols is None else n_cols
    table = {}
    for terms in pset.terminals.values():
        for term in terms:
            if not isinstance(term, gp.Terminal):
                table[term.name] = (K_CONST, None)
                continue
            i = pset.arguments.index(term.value)
            nm = str(term.value)
            if nm.startswith('CAT_'):
                table[term.name] = (K_CAT, FACTOR_COLUMNS.index(nm[4:]))
            elif nm.startswith('FAC_'):
                table[term.name] = (K_FAC, FACTOR_COLUMNS.index(nm[4:]))
            else:
                table[term.name] = (K_COL, i) if i < n_cols else (K_FAC, i - n_cols)
    for prims in pset.primitives.values():
        for prim in prims:
            table[prim.name] = (K_PRIM, prim.name)
    return table

def is_cat_gene(tree, table):
    return len(tree) == 1 and table.get(tree[0].name, (None,))[0] == K_CAT


# 나무를 (종류, 값) 배열로 바꾼다. forward가 이것을 읽는다.
def compile_tree(tree, table):
    T = len(tree)
    kind = [0] * T
    payload = [None] * T
    kids = [()] * T
    stack = []
    for i in range(T - 1, -1, -1):
        node = tree[i]
        k, v = table.get(node.name, (K_CONST, None))
        if k == K_CONST:
            kind[i], payload[i] = K_CONST, float(node.value)
        elif k == K_PRIM:
            kind[i], payload[i] = K_PRIM, v
            kids[i] = tuple(stack.pop() for _ in range(node.arity))
        else:
            kind[i], payload[i] = k, v
        stack.append(i)
    return kind, payload, kids

def _apply(name, args):
    if name == 'add':
        return args[0] + args[1]
    if name == 'sub':
        return args[0] - args[1]
    if name == 'mul':
        return args[0] * args[1]
    if name == 'div':
        return protected_div(args[0], args[1])
    if name == 'log':
        return protected_log(args[0])
    if name == 'sqrt':
        return protected_sqrt(args[0])
    if name == 'max':
        return np.maximum(args[0], args[1])
    if name == 'min':
        return np.minimum(args[0], args[1])
    raise KeyError(name)

def _partials(name, args):
    if name == 'add':
        return 1.0, 1.0
    if name == 'sub':
        return 1.0, -1.0
    if name == 'mul':
        return args[1], args[0]
    if name == 'div':
        a, b = args
        ok = np.abs(b) > PROTECT_EPS
        inv = np.where(ok, 1.0 / np.where(ok, b, 1.0), 0.0)
        return inv, np.where(ok, -a * inv * inv, 0.0)
    if name == 'log':
        a = args[0]
        ok = np.abs(a) > PROTECT_EPS
        return (np.where(ok, 1.0 / np.where(ok, a, 1.0), 0.0),)
    if name == 'sqrt':
        a = args[0]
        return (np.sign(a) / (2.0 * np.sqrt(np.maximum(np.abs(a), PROTECT_EPS))),)
    if name == 'max':
        a, b = args
        return (a >= b).astype(float), (a < b).astype(float)
    if name == 'min':
        a, b = args
        return (a <= b).astype(float), (a > b).astype(float)
    raise KeyError(name)


# Trees are evaluated column-wise over all rows at once, postfix order, one pass. This is the hot loop.
# 나무를 모든 행에 대해 열 단위로 한 번에 계산한다. 후위 순회 한 번이면 된다. 실행 시간을 가장 많이 쓰는 루프다.
class Evaluator:

    def __init__(self, X, codes, table):
        self.n = X.shape[0]
        self.cols = [np.ascontiguousarray(X[:, j]) for j in range(X.shape[1])]
        self.codes = [codes[c] for c in FACTOR_COLUMNS if c in (codes or {})]
        self.catcols = [c.astype(float) for c in self.codes]
        self.table = table
        self._ones = np.ones(self.n)
        self._rows = np.arange(self.n)

    def forward(self, tree, theta, keep=False):
        kind, payload, kids = compile_tree(tree, self.table)
        val = [None] * len(kind)
        with np.errstate(all='ignore'):
            for i in range(len(kind) - 1, -1, -1):
                k = kind[i]
                if k == K_COL:
                    val[i] = self.cols[payload[i]]
                elif k == K_FAC:
                    j = payload[i]
                    val[i] = theta[FACTOR_COLUMNS[j]][self.codes[j]]
                elif k == K_CAT:
                    val[i] = self.catcols[payload[i]]
                elif k == K_CONST:
                    val[i] = payload[i] * self._ones
                else:
                    val[i] = _apply(payload[i], [val[c] for c in kids[i]])
        out = val[0]
        out = out if isinstance(out, np.ndarray) and out.shape == (self.n,) else np.broadcast_to(out, (self.n,))
        return (out, (kind, payload, kids, val)) if keep else out

    def backward(self, tape, upstream, sink):
        kind, payload, kids, val = tape
        adj = [None] * len(kind)
        adj[0] = upstream
        with np.errstate(all='ignore'):
            for i in range(len(kind)):
                a = adj[i]
                if a is None:
                    continue
                k = kind[i]
                if k == K_PRIM:
                    ds = _partials(payload[i], [val[c] for c in kids[i]])
                    for c, d in zip(kids[i], ds):
                        contrib = a * d
                        adj[c] = contrib if adj[c] is None else adj[c] + contrib
                elif k == K_FAC:
                    sink(payload[i], a)

    def grad_sink(self, grads):
        def f(j, a):
            np.add.at(grads[FACTOR_COLUMNS[j]], self.codes[j], a)
        return f

    def jac_sink(self, J, base):
        rows = self._rows
        def f(j, a):
            np.add.at(J, (rows, base[j] + self.codes[j]), a)
        return f

## The decider — ARD scored by PRESS / 결정자 — PRESS로 채점하는 ARD

### ARD, PRESS and clipping / ARD · PRESS · 클리핑

**ARD 반복** : 유전자 열을 train 행 기준으로 표준화한 행렬 $Z$에서, 유전자별 사전 정밀도 $\lambda_j$와 잡음 정밀도 $a$를 번갈아 갱신한다.

$$\Sigma = (a Z^\top Z + \operatorname{diag}\lambda)^{-1},\quad \beta = a\,\Sigma Z^\top (y - \bar y),\quad \gamma_j = 1 - \lambda_j \Sigma_{jj},\quad \lambda_j \leftarrow \frac{\gamma_j}{\beta_j^2},\quad a \leftarrow \frac{n - \sum_j \gamma_j}{\text{RSS}}$$

$\gamma_j$는 데이터가 그 계수를 얼마나 확실하게 정해 주는지(0\~1)다. 이때 $\beta$는 피쳐마다 $\alpha_j = \lambda_j / a$를 쓰는 릿지의 해와 같다.<br>
$\lambda_j$가 한없이 커진 유전자는 계수가 0이 되어 꺼지고, 이후 계산에서 빠진다.<br>
보통 일곱 번 안팎에 수렴하고, 반복이 수치적으로 실패하면 α 하나짜리 릿지로 대신한다.

**PRESS 계산** : 마지막 반복의 $\Sigma$로 레버리지 $h_{ii} = 1/n + a\,z_i^\top \Sigma z_i$(절편 몫 + 유전자 몫)를 구하고, 한 번 맞춘 잔차 $e_i$로 $e_i/(1 - h_{ii})$를 모은다.<br>
계수를 $n$번 다시 풀 필요가 없다. PRESS를 쓰는 이유는 0절에 적었다.

**클리핑** : 적합할 때 유전자마다 train 행의 최솟값 · 최댓값(`g_lo`, `g_hi`)을 저장하고, `predict`가 새 행의 유전자 값을 그 범위로 자른다.<br>
예측값을 train 가격 범위로 자르는 일은 진화 루프의 valid 채점(`valid_rmse`)과 5절의 반환 모델 예측에서 한다.

In [16]:
# The decider: standardize the gene columns, solve ARD (one alpha per gene), score it by PRESS.
# 결정자: 유전자 열을 표준화하고 ARD(유전자마다 α 하나)를 풀어 PRESS로 채점한다.
#   fit_ard     : 증거 최대화로 유전자마다 α_j를 정한다. 쓸모없는 유전자는 계수 0으로 꺼진다
#   _press      : 한 행 빼기 오차 e_i / (1 - h_ii)의 RMS. 적합도의 오차 항이다
#   _bounds     : 유전자마다 train 행의 최솟값·최댓값. predict가 새 행의 유전자 값을 그 범위로 자른다
#   fit         : 단일 α 릿지(GCV로 고른다). 결정자가 아니다 — 릿지 89열 기준선과 ARD의 비상 채점에만 쓴다
#   gene_matrix : 개체의 나무들을 열로 쌓는다 (n × 유전자수)
#   refine      : theta 경사 정련. 이 노트북에서는 iters=0이라 적합만 한다


# Candidate alphas for the single-alpha ridge (fit). The decider is ARD alone — this ridge is used in two places only:
# the ridge-89 baseline (section 4, the ruler of the score gate) and the emergency fallback when an ARD solve breaks.
# 단일 α 릿지(fit)의 α 후보. 결정자는 ARD 하나다 — 이 릿지는 두 곳에만 쓴다.
# 4절의 릿지 89열 기준선(8절 점수 문턱의 기준), 그리고 ARD 풀이가 수치적으로 깨진 개체의 비상 채점.
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)

MIN_STD = 1e-12


# 분모(1 − h)의 하한. 레버리지가 1에 닿은 행에서도 0으로 나누지 않게 한다.
PRESS_EPS = 1e-12


# PRESS: every training row scored by the model fitted without it, e_i / (1 - h_ii). A row the gene memorised
# is scored as if the gene were absent, so memorising earns nothing.
# PRESS. train 행마다 그 행을 빼고 맞춘 모델의 오차 e_i / (1 - h_ii)를 쓴다. 외운 행은 그 유전자가 없을 때처럼 채점되므로 외워도 점수를 얻지 못한다.
def _press(e, h):
    if h is None or not np.all(np.isfinite(h)):
        return np.inf
    loo = e / np.maximum(1.0 - h, PRESS_EPS)
    v = float(np.sqrt(np.mean(loo * loo)))
    return v if np.isfinite(v) else np.inf


# 유전자마다 train 행의 최솟값·최댓값. 꺼진 유전자는 자를 필요가 없어 ±inf다.
def _bounds(Gmat, alive):
    lo = np.full(Gmat.shape[1], -np.inf)
    hi = np.full(Gmat.shape[1], np.inf)
    if alive.any():
        lo[alive] = Gmat[:, alive].min(axis=0)
        hi[alive] = Gmat[:, alive].max(axis=0)
    return lo, hi


# 유전자 열을 표준화한다. 분산이 거의 없거나 유한하지 않은 열은 alive=False로 떨어낸다.
def standardize(G):
    finite = np.isfinite(G).all(axis=0)
    mu = np.where(finite, G.mean(axis=0, where=np.isfinite(G)), 0.0)
    sd = np.where(finite, G.std(axis=0, where=np.isfinite(G)), 0.0)
    alive = finite & (sd > MIN_STD)
    sd_safe = np.where(alive, sd, 1.0)
    Z = (G - mu) / sd_safe
    Z[:, ~alive] = 0.0
    return Z, mu, sd_safe, alive


# Single-alpha ridge, alpha picked by GCV. Not a decider here — only the ridge-89 baseline and ARD's emergency fallback.
# 단일 α 릿지(GCV로 α를 고른다). 이 노트북의 결정자가 아니다 — 릿지 89열 기준선과 ARD의 비상 채점에만 쓴다.
def fit(Gmat, y, n_theta=0, alphas=ALPHAS):
    n = len(y)
    Z, mu, sd, alive = standardize(Gmat)
    k = int(alive.sum())
    D = np.empty((n, k + 1))
    D[:, 0] = 1.0
    if k:
        D[:, 1:] = Z[:, alive]
    G = D.T @ D
    c = D.T @ y
    yy = float(y @ y)
    pen = np.eye(k + 1)
    pen[0, 0] = 0.0
    best = None
    for a in alphas:
        A = G + a * pen
        try:
            b = np.linalg.solve(A, c)
            tr = float(np.trace(np.linalg.solve(A, G)))
        except np.linalg.LinAlgError:
            continue
        rss = yy - 2.0 * float(b @ c) + float(b @ G @ b)
        rss = max(rss, 0.0)
        df = tr + n_theta
        denom = 1.0 - df / n
        gcv = np.inf if denom <= 1e-6 else (rss / n) / (denom * denom)
        if not np.isfinite(gcv):
            gcv = np.inf
        if best is None or gcv < best[0]:
            best = (gcv, a, b, rss, df)
    if best is None:
        return None
    gcv, a, b, rss, df = best
    beta = np.zeros(Gmat.shape[1])
    beta[alive] = b[1:]
    try:
        h = np.einsum('ij,ji->i', D, np.linalg.solve(G + a * pen, D.T))
    except np.linalg.LinAlgError:
        h = None
    lo, hi = _bounds(Gmat, alive)
    return dict(alpha=a, beta=beta, intercept=float(b[0]), mu=mu, sd=sd, alive=alive,
                rss=rss, df=df, gcv_rmse=float(np.sqrt(gcv)), train_rmse=float(np.sqrt(rss / n)),
                press_rmse=_press(y - D @ b, h), g_lo=lo, g_hi=hi)


# ARD 반복 상한과 수렴 기준. 보통 일곱 번 안팎에 수렴한다.
ARD_ITERS = 100

ARD_TOL = 1e-6


# 사전 정밀도가 이보다 크면 그 유전자는 꺼진 것으로 보고 계산에서 뺀다.
ARD_CAP = 1e10


# 릿지 단위 α_j가 이보다 크면 꺼진 유전자로 센다(기록용).
ARD_OFF = 1e4


# ARD: a ridge with one penalty per gene, each set by maximising the evidence (MacKay 1992, Tipping 2001).
# A useless gene's penalty grows without bound and its coefficient becomes exactly 0.
# ARD. 유전자마다 벌점을 따로 두는 릿지이고, 각 벌점은 증거 최대화로 정한다. 쓸모없는 유전자는 벌점이 한없이 커져 계수가 0이 된다.
# 반환하는 β는 α_j = λ_j / a인 피쳐별 릿지의 해와 같다. PRESS의 레버리지도 이 α_j를 고정해 계산한다.
# If the solve breaks numerically (the except branch), that one individual is scored by fit and marked ard_fallback=True.
# 풀이가 수치적으로 깨지면(except 갈래) 그 개체 하나만 fit으로 채점하고 ard_fallback=True로 표시한다.
# 같은 셋업을 끝까지 돌린 런에서는 세대별 1등 1,559개 중 이렇게 채점된 개체가 없었다.
def fit_ard(Gmat, y, n_theta=0, iters=ARD_ITERS, tol=ARD_TOL):
    n = len(y)
    Z, mu, sd, alive = standardize(Gmat)
    idx = np.flatnonzero(alive)
    k = len(idx)
    yb = float(y.mean())
    yc = y - yb
    beta = np.zeros(Gmat.shape[1])
    alpha_j = np.full(Gmat.shape[1], np.inf)
    try:
        X = Z[:, idx]
        XtX, Xty = X.T @ X, X.T @ yc
        lam = np.ones(k)
        a = 1.0 / max(float(yc.var()), 1e-12)
        m = np.zeros(k)
        g = np.zeros(k)
        it = 0
        for it in range(1, iters + 1):
            act = lam < ARD_CAP
            mn, g = np.zeros(k), np.zeros(k)
            if act.any():
                S = np.linalg.inv(a * XtX[np.ix_(act, act)] + np.diag(lam[act]))
                mn[act] = a * S @ Xty[act]
                g[act] = 1.0 - lam[act] * np.diag(S)
            r = yc - X @ mn
            a = (n - g.sum()) / max(float(r @ r), 1e-30)
            lam = np.where(act, np.minimum(g / np.maximum(mn ** 2, 1e-300), ARD_CAP * 10), lam)
            done = np.max(np.abs(mn - m)) < tol if k else True
            m = mn
            if done:
                break
        act = lam < ARD_CAP
        m, g = np.zeros(k), np.zeros(k)
        h = np.full(n, 1.0 / n)
        if act.any():
            S = np.linalg.inv(a * XtX[np.ix_(act, act)] + np.diag(lam[act]))
            m[act] = a * S @ Xty[act]
            g[act] = 1.0 - lam[act] * np.diag(S)
            Xa = X[:, act]
            h = h + a * np.sum((Xa @ S) * Xa, axis=1)
        if not (np.all(np.isfinite(m)) and np.isfinite(a)):
            raise FloatingPointError('ARD가 유한하지 않다')
    except (np.linalg.LinAlgError, FloatingPointError):
        s = fit(Gmat, y, n_theta=n_theta)
        if s is not None:
            s.update(decider='ard', ard_fallback=True, alpha_j=np.where(s['alive'], s['alpha'], np.inf),
                     n_off=0, ard_iters=0)
        return s
    r = yc - X @ m
    rss = float(r @ r)
    df = float(g.sum()) + 1.0 + n_theta
    denom = 1.0 - df / n
    gcv = np.inf if denom <= 1e-6 else (rss / n) / (denom * denom)
    beta[idx] = m
    alpha_j[idx] = lam / a
    lo, hi = _bounds(Gmat, alive)
    return dict(alpha=None, alpha_j=alpha_j, beta=beta, intercept=yb, mu=mu, sd=sd, alive=alive, rss=rss, df=df,
                gcv_rmse=float(np.sqrt(gcv)) if np.isfinite(gcv) else np.inf, train_rmse=float(np.sqrt(rss / n)),
                press_rmse=_press(r, h), g_lo=lo, g_hi=hi,
                decider='ard', ard_fallback=False, n_off=int((alpha_j[idx] > ARD_OFF).sum()), ard_iters=it)


# 결정자 이름 → 선형 적합 함수. 이 노트북은 ARD 하나다.
def fitter_for(decider):
    if decider == 'ard':
        return fit_ard
    raise ValueError(f'선형 결정자가 아니다: {decider}')


# Prediction on new rows: each gene's output is first clipped to the range it had on the training rows.
# 새 행 예측. 유전자 값을 먼저 그 유전자의 train 범위로 자른다. train 행은 그대로라 ARD 해가 바뀌지 않는다.
def predict(Gmat, sol):
    if sol.get('g_lo') is not None:
        Gmat = np.clip(Gmat, sol['g_lo'], sol['g_hi'])
    Z = (Gmat - sol['mu']) / sol['sd']
    Z = np.where(np.isfinite(Z), Z, 0.0)
    Z[:, ~sol['alive']] = 0.0
    return sol['intercept'] + Z @ sol['beta']

def param_layout(n_levels):
    base, p = {}, 0
    for j, c in enumerate(FACTOR_COLUMNS):
        if c not in (n_levels or {}):
            continue
        base[j] = p
        p += n_levels[c]
    return base, p

def normalize(theta, out=None, center=0.0):
    out = out if out is not None else {}
    for c, v in theta.items():
        sd = float(v.std())
        out[c] = (v - v.mean()) / sd + center if sd > 1e-12 else np.full_like(v, center)
    return out


# theta가 쓰는 자유도. 이 노트북에서는 0이다.
def effective_df(n_levels):
    return int(sum(max(n - 2, 0) for n in n_levels.values()))

def flatten(theta):
    parts = [theta[c] for c in FACTOR_COLUMNS if c in theta]
    return np.concatenate(parts) if parts else np.zeros(0)

def unflatten(vec, n_levels):
    out, i = {}, 0
    for c in FACTOR_COLUMNS:
        if c not in (n_levels or {}):
            continue
        n = n_levels[c]
        out[c] = vec[i:i + n].copy()
        i += n
    return out


# The individual's trees become the columns of one matrix. This is what ARD reads.
# 개체의 나무들이 행렬 하나의 열이 된다. ARD는 이 행렬을 받는다.
def gene_matrix(ev, genes, theta, keep=False):
    cols, tapes = [], []
    for g in genes:
        if keep:
            v, tp = ev.forward(g, theta, keep=True)
            tapes.append(tp)
        else:
            v = ev.forward(g, theta)
        cols.append(v)
    G = np.column_stack(cols) if cols else np.zeros((ev.n, 0))
    return (G, tapes) if keep else G


# theta 경사 정련. 이 노트북에서는 iters=0이라 유전자 행렬을 만들고 결정자를 한 번 푼다.
def refine(ev, genes, theta, y, n_levels, iters=10, alpha_theta=0.0, lam=1e-3, n_theta_df=0,
           normalize_theta=True, center=0.0, fitter=None):
    fitter = fitter or fit
    base, p = param_layout(n_levels)
    if normalize_theta:
        theta = normalize(theta, center=center)
    theta0 = flatten(theta)
    cur = theta0.copy()
    curd = unflatten(cur, n_levels)
    G, tapes = gene_matrix(ev, genes, curd, keep=True)
    sol = fitter(G, y, n_theta=n_theta_df)
    if sol is None or iters == 0 or p == 0:
        return curd, sol, 0

    def objective(vec):
        td = unflatten(vec, n_levels)
        if normalize_theta:
            td = normalize(td, td, center=center)
        Gm, tp = gene_matrix(ev, genes, td, keep=True)
        s = fitter(Gm, y, n_theta=n_theta_df)
        if s is None:
            return np.inf, None, None, None
        r = y - predict(Gm, s)
        d = vec - theta0
        obj = 0.5 * float(r @ r) + 0.5 * alpha_theta * float(d @ d)
        return obj, s, r, tp

    obj, sol, r, tapes = objective(cur)
    accepted = 0
    for _ in range(iters):
        if not np.isfinite(obj):
            break
        J = np.zeros((ev.n, p))
        sink = ev.jac_sink(J, base)
        ones = np.ones(ev.n)
        for k, tp in enumerate(tapes):
            if not sol['alive'][k] or sol['beta'][k] == 0.0:
                continue
            ev.backward(tp, ones * (sol['beta'][k] / sol['sd'][k]), sink)
        if not np.isfinite(J).all():
            J = np.nan_to_num(J, nan=0.0, posinf=0.0, neginf=0.0)
        JtJ = J.T @ J
        g = J.T @ r - alpha_theta * (cur - theta0)
        D = np.diag(np.maximum(np.diag(JtJ), 1e-12))
        step = None
        try:
            step = np.linalg.solve(JtJ + alpha_theta * np.eye(p) + lam * D, g)
        except np.linalg.LinAlgError:
            lam *= 10
            continue
        if not np.isfinite(step).all():
            lam *= 10
            continue
        cand = cur + step
        new_obj, new_sol, new_r, new_tapes = objective(cand)
        if new_obj < obj:
            if normalize_theta:
                cand = flatten(normalize(unflatten(cand, n_levels), center=center))
            cur, obj, sol, r, tapes = cand, new_obj, new_sol, new_r, new_tapes
            lam = max(lam / 10, 1e-12)
            accepted += 1
        else:
            lam *= 10
            if lam > 1e12:
                break
    final = unflatten(cur, n_levels)
    return (normalize(final, center=center) if normalize_theta else final), sol, accepted

## The search — the evolution loop / 탐색 — 진화 루프

### **유전 프로그래밍이 도는 방식**
한 세대는 평가 → 순위 매기기 → 번식으로 한 바퀴 돈다.

- **초기화(0세대만)** : 모든 개체는 유전자 1개로 시작하고, 나무는 높이 2\~6에서 무작위로 만든다. 유전자 수는 진화하면서 늘어난다(최대 45개).
- **평가** : 유전자로 행렬을 만들고 ARD를 푼다. 적합도 = PRESS + 노드 가격 × 노드 수 + 유전자 가격 × 살아 있는 유전자 수. 유전자 구성이 부모와 같은 자식은 다시 풀지 않고 부모의 결과를 쓴다.
- **순위 매기기** : 적합도가 낮을수록 이긴다. 적합도가 같으면 짧은 프로그램이 이긴다.
- **번식** : 상위 15%는 그대로 다음 세대로 넘긴다(같은 모델은 한 번만). 나머지 자리는 토너먼트로 뽑은 부모 쌍의 자식으로 채운다 — 토너먼트는 무작위로 15개를 뽑아 그중 가장 좋은 개체를 부모로 삼는다.

뽑힌 부모 한 쌍은 교차나 복제를 거쳐 자식이 되고, 그 자식에 돌연변이가 일어날 수 있다.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">단계</th><th style="text-align:center">확률</th><th style="text-align:left">무엇이 바뀌는가</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>교차</b></td><td style="text-align:center">90%</td><td style="text-align:left">이 중 20%는 상위 교차 — 유전자마다 50% 확률로 상대에게 넘긴다. 유전자 수가 바뀐다.<br>나머지 80%는 하위 교차 — 양쪽에서 유전자를 하나씩 골라 가지를 맞바꾼다</td></tr>
<tr><td style="text-align:left"><b>복제</b></td><td style="text-align:center">10%</td><td style="text-align:left">두 부모를 그대로 복사한다</td></tr>
<tr><td style="text-align:left"><b>돌연변이</b></td><td style="text-align:center">자식마다 10%</td><td style="text-align:left">유전자 추가 · 삭제(정확히 1개) · 가지 바꾸기 중 하나.<br>10세대 전에는 0.5 · 0.1 · 0.4, 그 뒤에는 0.1 · 0.1 · 0.8</td></tr>
</tbody></table>
</div>

연산은 유전자 수를 어느 한쪽으로 밀지 않도록 짰다 — 유전자 수는 선택이 정한다.<br>
결과 나무의 높이가 6을 넘으면 그 결과는 버리고 부모 것을 둔다. 한 개체 안에 같은 식이 여러 번 나오면 하나만 남긴다.

선택압(좋은 개체가 부모로 얼마나 쏠려 뽑히는가)은 토너먼트 크기 15에서 나온다. 개체군이 10,000일 때 계층별 몫은 다음과 같다.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">계층</th><th style="text-align:center">개체 수</th><th style="text-align:center">부모 자리 중 지분</th><th style="text-align:center">개체당 기대 자손</th></tr></thead>
<tbody>
<tr><td style="text-align:left">1등</td><td style="text-align:center">1</td><td style="text-align:center">0.15%</td><td style="text-align:center">14.99</td></tr>
<tr><td style="text-align:left">상위 1%</td><td style="text-align:center">100</td><td style="text-align:center">13.99%</td><td style="text-align:center">13.99</td></tr>
<tr><td style="text-align:left">상위 5%</td><td style="text-align:center">500</td><td style="text-align:center">53.67%</td><td style="text-align:center">10.73</td></tr>
<tr><td style="text-align:left">상위 10%</td><td style="text-align:center">1,000</td><td style="text-align:center">79.41%</td><td style="text-align:center">7.94</td></tr>
<tr><td style="text-align:left"><b>상위 20%</b></td><td style="text-align:center">2,000</td><td style="text-align:center"><b>96.48%</b></td><td style="text-align:center">4.82</td></tr>
<tr><td style="text-align:left">하위 50%</td><td style="text-align:center">5,000</td><td style="text-align:center">0.003%</td><td style="text-align:center">0.00</td></tr>
</tbody></table>
</div>

상위 20%가 부모 자리의 96.48%(1 − 0.8¹⁵)를 차지하고, 하위 절반의 몫은 0.5¹⁵ = 0.003%로 사실상 0이다. 여기에 상위 15%가 엘리트로 그대로 넘어간다.<br>
상위 20%만 잘라 부모로 쓰는 절단 선택도 선택의 세기는 같지만 다양성을 잃는다 — 토너먼트에서는 순위가 낮은 개체도 뽑힐 기회가 남는다(Blickle &amp; Thiele, *Evolutionary Computation* 4(4), 1996).

**가격 일정** : 유전자 가격은 개체군이 다 자랄 때까지 0이고, 그 뒤 250세대에 걸쳐 0.003까지 오른다. 키우고 깎는 과정과 멈출 자리를 고르는 법은 0절에 적었다.

### Evolution loop / 진화 루프

진화는 train 행만 본다.<br>
valid 행은 세대마다 1등을 한 번 채점하는 데만 쓰고, 그 기록으로 5절의 통합 valid 최적이 돌려줄 세대를 고른다.<br>
그래서 세대마다 1등을 후보로 저장해 둔다(`ckpt` 콜백).

In [17]:
# The evolution loop: evaluate → rank → breed, with the gene price that turns growth into pruning.
# 진화 루프. 평가 → 순위 → 번식을 반복하고, 유전자 가격이 붙으면서 키우기가 깎기로 바뀐다.
#   assign_fitness : PRESS + 노드 가격 × 노드 수 + 유전자 가격 × 살아 있는 유전자 수
#   grown, gene_price : 다 자람 판정과 그 뒤 250세대 동안 0 → 0.003으로 오르는 유전자 가격
#   select_elite, hi_crossover, mutate : 엘리트 15%, 상위·하위 교차, 돌연변이(유전자 추가·삭제·가지 바꾸기)
#   run            : 세대 루프. 세대마다 상위 개체를 ckpt 콜백에 넘긴다 — 1등은 통합 valid 최적(5절)의 후보다
#   cfg['max_seconds'] : 실행 시간 상한. Kaggle 세션 시간 안에서 깔끔하게 멈춘다


# 기본 설정. 노트북의 CFG가 이 셋업의 값을 덮어쓴다.
DEFAULTS = dict(population=10000, genes=45, variable_genes=True, init_genes=(1, 1),
                max_height=6, init_height=(2, 6), mut_height=(0, 2),
                parsimony=1e-5, tournament=15,
                elite_frac=0.15,
                p_crossover=0.9,
                p_cross_hi=0.2, hi_cross_rate=0.5,
                p_mutation=0.10,
                mut_mix_early=(0.5, 0.1, 0.4),
                mut_mix_late=(0.1, 0.1, 0.8),
                mut_early_gens=10,
                return_smooth=11,
                gene_price_max=0.003,
                gene_price_ramp=250,
                grow_window=10, grow_flat=0.5,
                grow_min=30, grow_max=150,
                alpha_theta=0.0, lm_early=10, lm_early_gens=10, lm_late=1,
                max_gen=400, seed=42,
                normalize_theta=True, theta_center='auto',
                covariant_parsimony=False, parsimony_scale=1.0, size_measure='linear',
                parsimony_decay='sqrt_gen', parsimony_target=None,
                decider='ridge', tree_depth=6,
                fitness_metric='press',
                clip_genes=True,
                es_patience=0, es_min_gens=10, max_seconds=0,
                ckpt_top=0,
                gbm_iters=50, gbm_depth=3, gbm_lr=0.1, gbm_bins=32)


# 개체 하나. 유전자 목록과 결정자 해, 그리고 계보(uid·parents·op)를 들고 있다.
class Ind:
    __slots__ = ('genes', 'theta', 'fit', 'sol', 'nodes', 'n_theta', 'uid', 'parents', 'op')


    def __init__(self, genes, theta):
        self.genes, self.theta = genes, theta
        self.fit = np.inf
        self.sol = None
        self.nodes = sum(len(g) for g in genes)
        self.n_theta = sum(len(v) for v in theta.values())
        self.uid, self.parents, self.op = 0, (), 'init'

    def copy(self):
        c = Ind(list(self.genes), {k: v.copy() for k, v in self.theta.items()})
        c.fit, c.sol = self.fit, self.sol
        c.uid, c.parents, c.op = self.uid, self.parents, self.op
        return c


# 같은 식인지 가리는 키. 한 개체 안의 중복 유전자를 줄이는 데 쓴다.
def gene_key(tree):
    k = getattr(tree, '_gk', None)
    if k is None:
        k = tuple(n.format() if n.arity == 0 else n.name for n in tree)
        try:
            _gk = k
        except AttributeError:
            pass
    return k


# 개체 전체의 키 = 유전자 키들을 정렬한 튜플. 결정자는 열 순서와 상관없으므로 유전자 집합이 같으면 같은 모델이다.
def genome_key(ind):
    k = tuple(sorted(gene_key(g) for g in ind.genes))
    if ind.theta:
        k = (k, tuple((c, np.asarray(v, float).tobytes()) for c, v in sorted(ind.theta.items())))
    return k


# High-level crossover: each gene passes to the other parent with probability 0.5 — this is how gene counts change.
# 상위 교차. 유전자마다 0.5 확률로 상대에게 넘긴다. 유전자 수가 바뀌는 교차다.
# 받는 쪽에 같은 식이 이미 있으면 넘기지 않는다. 넘겨도 중복 제거가 곧바로 지워서 유전자 수만 티 나지 않게 줄어든다.
def hi_crossover(ga, gb, K, rate):
    sa = [random.random() < rate for _ in ga]
    sb = [random.random() < rate for _ in gb]
    if not any(sa):
        sa[random.randrange(len(ga))] = True
    if not any(sb):
        sb[random.randrange(len(gb))] = True
    rem_a = [g for g, s in zip(ga, sa) if not s]
    rem_b = [g for g, s in zip(gb, sb) if not s]
    keys_a = {gene_key(g) for g in rem_a}
    keys_b = {gene_key(g) for g in rem_b}
    move_a, stay_a, move_b, stay_b = [], [], [], []
    for g, s in zip(ga, sa):
        if s:
            (stay_a if gene_key(g) in keys_b else move_a).append(g)
    for g, s in zip(gb, sb):
        if s:
            (stay_b if gene_key(g) in keys_a else move_b).append(g)
    c1 = rem_a + stay_a + move_b
    c2 = rem_b + stay_b + move_a
    return c1[:K], c2[:K], int(len(c1) > K) + int(len(c2) > K)


# 유전자 집합이 부모와 같아진 자식은 부모의 평가 결과를 그대로 쓴다. 결정자는 같은 입력에 늘 같은 답을 내므로 다시 풀어도 결과가 같다.
def reuse_sol(p, cands):
    pk = genome_key(p)
    for q, qk in cands:
        if pk == qk and q.sol is not None:
            p.genes, p.nodes, p.sol = list(q.genes), q.nodes, q.sol
            return True
    p.sol = None
    return False


# 적합도 순(같으면 노드가 적은 쪽)으로 상위 15%. 같은 모델은 한 번만 넣는다.
def select_elite(pop, fit, length, n, keys):
    out, seen = [], set()
    for j in np.lexsort((length, fit)):
        j = int(j)
        if pop[j].sol is None or not np.isfinite(fit[j]):
            continue
        k = keys[j]
        if k in seen:
            continue
        seen.add(k)
        out.append(pop[j])
        if len(out) >= n:
            break
    return out

def _new_gene(pset, cfg):
    if cfg.get('p_cat_gene', 0.0) and random.random() < cfg['p_cat_gene']:
        return gp.PrimitiveTree([random.choice(pset.terminals[Cat])])
    return gp.PrimitiveTree(gp.genHalfAndHalf(pset, cfg['init_height'][0], cfg['init_height'][1]))


# 한 개체 안에서 같은 식이 둘 이상이면 하나로 줄인다. 빈 자리는 채우지 않고 지운다.
def dedup(ind, pset, cfg):
    seen, out = set(), []
    for g in ind.genes:
        k = gene_key(g)
        if k in seen:
            if cfg['variable_genes']:
                continue
            g = _new_gene(pset, cfg)
            k = gene_key(g)
        seen.add(k)
        out.append(g)
    if not out:
        out = [_new_gene(pset, cfg)]
    ind.genes = out
    ind.nodes = sum(len(g) for g in out)
    return len(ind.genes)


# One mutation: add a gene, delete exactly one, or replace a subtree. None of them pushes the gene count either way.
# 돌연변이는 셋 중 하나다 — 유전자 추가, 정확히 하나 삭제, 가지 바꾸기. 어느 것도 유전자 수를 한쪽으로 치우치게 하지 않는다.
def mutate(p, mix, K, pset, cfg, mut_expr, table):
    u = random.random()
    if u < mix[0] and len(p.genes) < K:
        p.genes = p.genes + [_new_gene(pset, cfg)]
        return 'gene+'
    if mix[0] <= u < mix[0] + mix[1] and len(p.genes) > 1:
        i = random.randrange(len(p.genes))
        p.genes = p.genes[:i] + p.genes[i + 1:]
        return 'gene-'
    i = random.randrange(len(p.genes))
    if cfg.get('p_cat_gene') and is_cat_gene(p.genes[i], table) and random.random() < 0.5:
        p.genes[i] = _new_gene(pset, cfg)
        return 'mut_cat'
    g, = gp.mutUniform(gp.PrimitiveTree(p.genes[i]), expr=mut_expr, pset=pset)
    if tree_height(g) <= cfg['max_height']:
        p.genes[i] = g
    return 'mut_sub'

def evaluate(ind, ev, y, n_levels, cfg, iters, bag_seed=0, ev_va=None, yv=None):
    df = effective_df(n_levels) if cfg['normalize_theta'] else ind.n_theta
    theta, sol, _ = refine(ev, ind.genes, ind.theta, y, n_levels, iters=iters,
                              alpha_theta=cfg['alpha_theta'], n_theta_df=df,
                              normalize_theta=cfg['normalize_theta'], center=cfg['theta_center'],
                              fitter=fitter_for(cfg.get('decider', 'ridge')))
    if sol is None:
        ind.fit, ind.sol = np.inf, None
        return
    if not cfg.get('clip_genes', False):
        sol.pop('g_lo', None)
        sol.pop('g_hi', None)
    ind.theta, ind.sol = theta, sol


# 적합도의 오차 항으로 쓸 키. 이 셋업은 PRESS다.
def err_key(cfg):
    return 'press_rmse' if cfg.get('fitness_metric', 'gcv') == 'press' else 'gcv_rmse'


# 개체군 전부의 부모-자식 간선. 계보 그래프의 재료다.
def _lineage(pop, gen):
    out = []
    for i in pop:
        g = i.sol['gcv_rmse'] if i.sol is not None else float('inf')
        out.append((gen, i.uid,
                    i.parents[0] if i.parents else -1,
                    i.parents[1] if len(i.parents) > 1 else -1,
                    i.op, i.fit, g, i.nodes, len(i.genes)))
    return out


# 세대마다 어떤 터미널·연산자·식이 얼마나 쓰였는지 센다. 잔존율 그림의 재료다.
def _census(pop, top_n=20):
    terms, prims, exprs = Counter(), Counter(), Counter()
    n = 0
    for ind in pop:
        if ind.sol is None:
            continue
        for g in ind.genes:
            n += 1
            exprs[str(g)] += 1
            for node in g:
                if isinstance(node, gp.Primitive):
                    prims[node.name] += 1
                elif isinstance(getattr(node, 'value', None), str):
                    terms[node.value] += 1
                else:
                    terms['상수'] += 1
    n_ind = sum(1 for i in pop if i.sol is not None)
    return dict(terminals=terms, prims=prims, exprs=exprs.most_common(top_n),
                n_genes=n, n_ind=n_ind)


# 개체군 수준 통계(분위수·다양성·PRESS). 결정자가 이미 계산한 값을 모으기만 한다.
def _pop_stats(pop, best):
    sols = [i.sol for i in pop if i.sol is not None]
    if not sols:
        return dict(pop_min_gcv=np.inf, avg_trees=np.nan, cat_share=np.nan,
                    avg_cat=np.nan, uniq_gcv=0, best_trees=np.nan, best_cat=np.nan,
                    press_q50=np.nan, pop_min_press=np.nan,
                    **{f'gcv_q{k:02d}': np.nan for k in (5, 25, 50, 75, 95)})
    gcv = np.array([s['gcv_rmse'] for s in sols], float)
    press = np.array([s.get('press_rmse', np.nan) for s in sols], float)
    pf = press[np.isfinite(press)]
    trees = [s.get('trees') for s in sols if s.get('trees') is not None]
    ncat = [s.get('n_cat') for s in sols if s.get('n_cat') is not None]
    ngen = [len(i.genes) for i in pop if i.sol is not None]
    bs = best.sol or {}
    q = np.percentile(gcv[np.isfinite(gcv)], [5, 25, 50, 75, 95]) if np.isfinite(gcv).any() \
        else [np.nan] * 5
    return dict(
        gcv_q05=float(q[0]), gcv_q25=float(q[1]), gcv_q50=float(q[2]),
        gcv_q75=float(q[3]), gcv_q95=float(q[4]),
        pop_min_gcv=float(gcv.min()),
        uniq_gcv=int(len(np.unique(np.round(gcv, 9)))),
        press_q50=(float(np.median(pf)) if pf.size else np.nan),
        pop_min_press=(float(pf.min()) if pf.size else np.nan),
        avg_trees=(float(np.mean(trees)) if trees else np.nan),
        avg_cat=(float(np.mean(ncat)) if ncat else np.nan),
        cat_share=(float(np.sum(ncat) / np.sum(ngen)) if ncat and np.sum(ngen) else np.nan),
        best_trees=float(bs.get('trees', np.nan)),
        best_cat=float(bs.get('n_cat', np.nan)))

def _size(nodes, measure):
    return np.sqrt(nodes) if measure == 'sqrt' else nodes


# 공변 절약 벌점 계수. 이전 런에서 썼고, 이 셋업은 고정 노드 가격을 쓰므로 부르지 않는다.
def parsimony_coefficient(pop, scale=1.0, measure='linear', key='gcv_rmse'):
    e = np.array([i.sol[key] for i in pop if i.sol is not None], float)
    l = _size(np.array([i.nodes for i in pop if i.sol is not None], float), measure)
    if len(e) < 10:
        return 0.0
    var = float(np.var(l))
    if var <= 0 or not np.isfinite(var):
        return 0.0
    c = -float(np.cov(l, e)[0, 1]) / var
    return max(0.0, c) * scale


# 결정자가 실제로 쓰는 유전자(열) 수. 퇴화 유전자는 빠진다. 유전자 가격은 이 수에 붙는다.
def n_alive(ind):
    s = ind.sol
    return int(np.asarray(s['alive']).sum()) if s is not None and 'alive' in s else len(ind.genes)


# The gene price: 0 until the population is grown, then rising linearly to 0.003 over 250 generations.
# 유전자 가격. 다 자랄 때까지 0이고, 그 뒤 250세대에 걸쳐 0.003까지 일정하게 오른다. 최댓값에 닿으면 런이 끝난다.
def gene_price(gen, t0, cfg):
    if t0 is None or gen <= t0:
        return 0.0
    return float(cfg['gene_price_max']) * min(1.0, (gen - t0) / max(1, int(cfg['gene_price_ramp'])))


# 다 자람 판정. 개체군 평균 유전자 수가 10세대 동안 0.5개도 변하지 않으면 다 자란 것으로 본다(30세대 전에는 판정하지 않고, 150세대가 되면 무조건 판정한다).
def grown(avg_k, gen, cfg):
    if gen < int(cfg['grow_min']):
        return False
    if gen >= int(cfg['grow_max']):
        return True
    w = int(cfg['grow_window'])
    return gen >= w and abs(avg_k[gen] - avg_k[gen - w]) < float(cfg['grow_flat'])


# Fitness = PRESS + node price x nodes + gene price x live genes. Lower is better.
# 적합도 = PRESS + 노드 가격 × 노드 수 + 유전자 가격 × 살아 있는 유전자 수. 낮을수록 좋다.
def assign_fitness(pop, df, cfg, gen=0, state=None, price=0.0):
    m = cfg['size_measure']
    key = err_key(cfg)
    if not cfg['covariant_parsimony']:
        c = cfg['parsimony']
    else:
        raw = parsimony_coefficient(pop, 1.0, m, key)
        mode = cfg['parsimony_decay']
        if mode == 'sqrt_gen':
            c = raw * cfg['parsimony_scale'] / np.sqrt(max(gen, 1))
        elif mode == 'target' and cfg['parsimony_target']:
            st = state if state is not None else {}
            mean_size = float(np.mean([i.nodes for i in pop]))
            g = st.get('gain', 1.0)
            g = float(np.clip(g * np.clip(mean_size / cfg['parsimony_target'], 1 / 1.3, 1.3), 1e-3, 1e3))
            st['gain'] = g
            c = raw * g * cfg['parsimony_scale']
        else:
            c = raw * cfg['parsimony_scale']
    for ind in pop:
        ind.fit = (np.inf if ind.sol is None
                   else ind.sol[key] + c * (_size(ind.nodes, m) + df) + price * n_alive(ind))
    return c


# 개체의 식과 학습에서 구한 결정자 해로 다른 행 묶음을 예측한다.
def predict_on(ind, ev):
    if ind.sol is None:
        return None
    pred = predict(gene_matrix(ev, ind.genes, ind.theta), ind.sol)
    return pred if np.isfinite(pred).all() else None

def rmse(pred, y):
    if pred is None:
        return np.inf
    d = pred - y
    return float(np.sqrt(d @ d / len(d)))


# 세대마다 1등을 valid 행으로 한 번 잰다. 예측은 fold의 train 가격 범위로 자른다.
def valid_rmse(ind, ev_valid, y_valid, lo=None, hi=None):
    pred = predict_on(ind, ev_valid)
    if pred is not None and lo is not None:
        pred = np.clip(pred, lo, hi)
    return rmse(pred, y_valid)


# 1등의 계수 진단: 가장 큰 |β|와 유전자 출력 사이 가장 큰 |상관|.
def best_diag(ind, ev):
    s = ind.sol
    if s is None or 'beta' not in s:
        return np.nan, np.nan
    b = np.abs(np.asarray(s['beta'], float))
    mb = float(b.max()) if b.size else np.nan
    alive = np.asarray(s['alive'], bool)
    if alive.sum() < 2:
        return mb, np.nan
    G = gene_matrix(ev, ind.genes, ind.theta)[:, alive]
    with np.errstate(invalid='ignore', divide='ignore'):
        R = np.corrcoef(G.T)
    R = np.abs(R[np.triu_indices(R.shape[0], 1)])
    R = R[np.isfinite(R)]
    return mb, (float(R.max()) if R.size else np.nan)


# The generation loop. Selection pressure comes from the training fitness only; the validation rows are scored once
# per generation on the champion, and the champion goes to the ckpt callback as a candidate (section 5).
# 세대 루프. 선택압은 학습 적합도에서만 나온다. valid 행은 세대마다 1등을 한 번 재는 데만 쓰이고,
# 그 1등은 ckpt 콜백으로 넘어가 통합 valid 최적(5절)의 후보가 된다.
def run(design, cfg=None, log=None, ckpt=None):
    cfg = dict(DEFAULTS, **(cfg or {}))
    random.seed(cfg['seed'])
    np.random.seed(cfg['seed'])
    rng = np.random.default_rng(cfg['seed'])
    pset = make_pset(getattr(design, 'arg_names', None))
    table = node_table(pset, getattr(design, 'n_cols', None))
    ev = Evaluator(design.X_train, design.codes_train, table)
    ev_va = (Evaluator(design.X_valid, design.codes_valid, table)
             if design.X_valid is not None else None)
    y, yv = design.y_train, design.y_valid
    y_lo, y_hi = float(np.min(y)), float(np.max(y))
    N, K = int(cfg['population']), int(cfg['genes'])
    if cfg['theta_center'] == 'auto':
        cfg['theta_center'] = float(np.median(design.X_train.mean(axis=0)))
    mate = gp.cxOnePoint

    def mut_expr(pset, type_=None):
        if type_ is not None and not pset.primitives.get(type_):
            return [random.choice(pset.terminals[type_])]
        return gp.genFull(pset, cfg['mut_height'][0], cfg['mut_height'][1], type_)

    n_cat = len(pset.terminals.get(Cat, ()))
    cfg.setdefault('p_cat_gene', n_cat / len(pset.arguments) if n_cat else 0.0)
    t0 = time.perf_counter()

    def fresh():
        g_lo, g_hi = cfg['init_genes']
        n = random.randint(int(g_lo), min(int(g_hi), K)) if cfg['variable_genes'] else K
        genes = [_new_gene(pset, cfg) for _ in range(n)]
        theta = {c: v + rng.normal(0, 0.1, size=v.shape) for c, v in design.theta_init.items()}
        if cfg['normalize_theta']:
            theta = normalize(theta, center=cfg['theta_center'])
        return Ind(genes, theta)

    nuid = [0]

    def _tag(ind, parents, op):
        nuid[0] += 1
        ind.uid, ind.parents, ind.op = nuid[0], tuple(parents), op
        return ind

    pop = [_tag(fresh(), (), 'init') for _ in range(N)]
    for ind in pop:
        dedup(ind, pset, cfg)
    rows = []
    pstate = {}
    stop = 'max_gen'
    nbag = [0]
    val_best = None
    hist = []
    avg_k = []
    grow_stop = None
    end_gen = None
    for gen in range(int(cfg['max_gen']) + 1):
        iters = cfg['lm_early'] if gen < cfg['lm_early_gens'] else cfg['lm_late']
        for ind in pop:
            if ind.sol is None:
                nbag[0] += 1
                evaluate(ind, ev, y, design.n_levels, cfg, iters,
                         bag_seed=nbag[0] + 7919 * cfg['seed'], ev_va=ev_va, yv=yv)
        price = gene_price(gen, grow_stop, cfg)
        c_used = assign_fitness(pop, effective_df(design.n_levels), cfg, gen, pstate, price)
        fit = np.array([i.fit for i in pop])
        length = np.array([i.nodes for i in pop], dtype=np.int64)
        b = int(np.lexsort((length, fit))[0])
        best = pop[b]
        keys = [genome_key(i) for i in pop]
        kcount = np.array([len(i.genes) for i in pop])
        mb, mc = best_diag(best, ev)
        row = dict(gen=gen, best_fit=best.fit, best_train=best.sol['train_rmse'] if best.sol else np.inf,
                   best_gcv=best.sol['gcv_rmse'] if best.sol else np.inf,
                   best_press=float(best.sol.get('press_rmse', np.nan)) if best.sol else np.inf,
                   best_valid=valid_rmse(best, ev_va, yv, y_lo, y_hi) if ev_va is not None else np.nan,
                   best_max_beta=mb, best_max_corr=mc,
                   pop_unique=int(len(set(keys))),
                   k_min=int(kcount.min()), k_q50=float(np.median(kcount)), k_max=int(kcount.max()),
                   **{f'bred_{k}': v for k, v in pstate.pop('bred', {}).items()},
                   best_nodes=best.nodes, best_alive=int(best.sol['alive'].sum()) if best.sol else 0,
                   best_genes=len(best.genes), avg_genes=float(np.mean([len(i.genes) for i in pop])),
                   best_alpha=(best.sol['alpha'] if best.sol and best.sol.get('alpha') is not None else np.nan),
                   best_df=(float(best.sol['df']) if best.sol and 'df' in best.sol else np.nan),
                   best_off=(int(best.sol.get('n_off', 0)) if best.sol else 0),
                   avg_off=float(np.mean([i.sol.get('n_off', 0) for i in pop if i.sol is not None] or [0])),
                   avg_fit=float(np.mean(fit[np.isfinite(fit)])) if np.isfinite(fit).any() else np.inf,
                   parsimony_c=c_used, gene_price=price, avg_nodes=float(np.mean([i.nodes for i in pop])),
                   n_inf=int((~np.isfinite(fit)).sum()), lm_iters=iters,
                   elapsed=time.perf_counter() - t0,
                   **_pop_stats(pop, best))
        if ev_va is not None and np.isfinite(row['best_valid']):
            if val_best is None or row['best_valid'] < val_best[0]:
                val_best = (row['best_valid'], gen, best.copy())
        row['val_best_so_far'] = val_best[0] if val_best else np.nan
        avg_k.append(row['avg_genes'])
        if grow_stop is None and grown(avg_k, gen, cfg):
            grow_stop = gen
            end_gen = gen + max(1, int(cfg['gene_price_ramp']))
        row['grow_stop'] = grow_stop if grow_stop is not None else np.nan
        rows.append(row)
        hist.append(best.copy())
        if ckpt and int(cfg.get('ckpt_top', 0)):
            order = np.lexsort((length, fit))[:int(cfg['ckpt_top'])]
            top, vs = [], []
            for rank, j in enumerate(order):
                ind = pop[int(j)]
                if ind.sol is None:
                    continue
                if rank == 0:
                    v = row['best_valid']
                else:
                    v = valid_rmse(ind, ev_va, yv, y_lo, y_hi) if ev_va is not None else np.nan
                top.append(ind)
                vs.append(v)
            ckpt(gen, top, vs, val_best, dict(
                census=(_census(pop, int(cfg.get('census_top', 20)))
                        if cfg.get('census_top', 20) else None),
                lineage=(_lineage(pop, gen) if cfg.get('lineage', 1) else None)))
        if log:
            log(row)
        if end_gen is not None and gen >= end_gen:
            stop = 'ramp_done'
            break
        if gen == int(cfg['max_gen']):
            break
        if cfg.get('max_seconds') and (time.perf_counter() - t0) > float(cfg['max_seconds']):
            stop = 'time'
            break
        if (cfg.get('es_patience') and val_best is not None
                and gen >= int(cfg.get('es_min_gens', 0))
                and gen - val_best[1] >= int(cfg['es_patience'])):
            stop = 'patience'
            break
        n_elite = max(1, int(round(float(cfg['elite_frac']) * N)))
        elites = [_tag(e.copy(), (e.uid,), 'elite')
                  for e in select_elite(pop, fit, length, n_elite, keys)]
        n_child = N - len(elites)
        n_pair = (n_child + 1) // 2
        parents = tournament(fit, length, 2 * n_pair, int(cfg['tournament']), rng)
        rows[-1]['sel_k'] = float(kcount[parents].mean() - kcount.mean())
        rows[-1]['sel_nodes'] = float(length[parents].mean() - length.mean())
        mix = cfg['mut_mix_early'] if gen < int(cfg['mut_early_gens']) else cfg['mut_mix_late']
        st = Counter()
        children = []
        for a, c in zip(parents[0::2], parents[1::2]):
            A, B = pop[a], pop[c]
            p1, p2 = A.copy(), B.copy()
            if random.random() < cfg['p_crossover']:
                if random.random() < cfg['p_cross_hi'] and (len(A.genes) > 1 or len(B.genes) > 1):
                    breed = 'xhi'
                    p1.genes, p2.genes, ntr = hi_crossover(A.genes, B.genes, K, cfg['hi_cross_rate'])
                    st['xhi_trunc'] += ntr
                else:
                    breed = 'xlo'
                    i1 = random.randrange(len(p1.genes))
                    i2 = random.randrange(len(p2.genes))
                    g1, g2 = mate(gp.PrimitiveTree(p1.genes[i1]), gp.PrimitiveTree(p2.genes[i2]))
                    if tree_height(g1) <= cfg['max_height']:
                        p1.genes[i1] = g1
                    if tree_height(g2) <= cfg['max_height']:
                        p2.genes[i2] = g2
            else:
                breed = 'repro'
            for p, cands in ((p1, ((A, keys[a]), (B, keys[c]))), (p2, ((B, keys[c]), (A, keys[a])))):
                mop = (mutate(p, mix, K, pset, cfg, mut_expr, table)
                       if random.random() < cfg['p_mutation'] else '')
                p.nodes = sum(len(g) for g in p.genes)
                dedup(p, pset, cfg)
                if reuse_sol(p, cands):
                    st['cached'] += 1
                op = breed + (f'|{mop}' if mop else '')
                _tag(p, (A.uid, B.uid), op)
                st[breed] += 1
                if mop:
                    st[mop] += 1
                children.append(p)
        children = children[:n_child]
        pop = elites + children
        pstate['bred'] = dict(elite=len(elites), repro=st['repro'], xhi=st['xhi'], xlo=st['xlo'],
                              gene_add=st['gene+'], gene_del=st['gene-'], mut_sub=st['mut_sub'],
                              mut_cat=st['mut_cat'], cached=st['cached'], xhi_trunc=st['xhi_trunc'])
    ma_best = None
    if ev_va is not None and rows:
        vv = np.array([r['best_valid'] for r in rows], float)
        w = max(1, int(cfg['return_smooth']))
        h = w // 2
        sm = np.array([np.nanmean(vv[max(0, i - h):i + h + 1]) for i in range(len(vv))])
        if np.isfinite(sm).any():
            g_ma = int(np.nanargmin(sm))
            ma_best = (float(vv[g_ma]), g_ma, hist[g_ma], float(sm[g_ma]))
    return dict(rows=rows, best=hist[-1] if hist else pop[0], pop=pop, stop=stop,
                val_best=val_best, ma_best=ma_best, last_valid=rows[-1]['best_valid'],
                grow_stop=grow_stop, runtime=time.perf_counter() - t0)

### Ridge-89 baseline / 릿지 89열 기준선

In [18]:
# A plain ridge on all 89 terminals, fold by fold. This is the bar the GP has to clear, and the
# score gate in section 8 compares against it — same rows, same split, so it is a fair ruler.
# 터미널 89개를 그대로 릿지에 넣었을 때의 점수다. GP가 넘어야 하는 기준선이고, 8절의 점수 문턱이 이 값과
# 비교한다. 같은 행·같은 분할에서 재므로 분할 난이도가 달라져도 기준이 흔들리지 않는다.
RIDGE89 = {}
print(f'{"fold":>5}{"alpha":>9}{"valid":>9}')
for f in range(5):
    d = DES[f]
    _sol = fit(d.X_train, d.y_train)
    RIDGE89[f] = float(np.sqrt(np.mean((predict(d.X_valid, _sol) - d.y_valid) ** 2)))
    print(f'{f:>5}{_sol["alpha"]:>9}{RIDGE89[f]:>9.4f}')

print(f'\nridge-89 mean valid {np.mean(list(RIDGE89.values())):.4f} — the GP has to beat this')

 fold    alpha    valid
    0     10.0   0.1384
    1     10.0   0.1329
    2     10.0   0.1293
    3     10.0   0.1237
    4     10.0   0.1208

ridge-89 mean valid 0.1290 — the GP has to beat this


# 5. Train / 학습

fold 5개를 워커 5개로 동시에 돌린다.<br>
워커는 `fork`로 설계행렬을 물려받아 큰 데이터를 피클로 주고받지 않고, 돌려받는 것은 fold별 결과뿐이다.

- **시간** : 본 런은 fold마다 개체군 10,000으로 돌고, 유전자 가격이 최댓값에 닿는 300\~320세대에서 끝난다. 데스크톱에서 fold 하나에 코어 하나씩 주면 fold당 41\~57분이 걸렸다. Kaggle 4코어에서는 7\~9시간으로 잡는다 — 이전 런이 Kaggle에서 데스크톱보다 5\~8배 느렸고, 이번에는 fold 다섯이 코어 넷을 나눠 쓴다.
- **시간 상한** : 세션 시간이 모자라면 `MAX_HOURS`가 fold를 깔끔하게 멈춘다. 멈춘 fold도 그때까지의 세대별 1등을 모두 돌려준다. 판정 뒤 11세대만 지나면 돌려줄 세대를 고를 수 있다 — 같은 셋업을 끝까지 돌린 런의 곡선(0절 그림 2)은 판정 +11 \~ +250 어디서 잘라도 같은 위치(판정 +5)를 골랐다.
- **계보** : 세대마다 정수 · 실수 배열로 바로 줄여 쌓는다. 간선 하나에 29바이트라, fold당 300만 간선이면 90MB 안팎이다.

In [19]:
# One worker per fold, all five at once. fork inherits DES/CFG, so only the result is pickled back.
# fold마다 워커 하나씩, 다섯 개를 동시에 돌린다. fork로 DES·CFG를 물려받으므로 결과만 피클로 돌려받는다.
import multiprocessing as mp

VIZ = OUTPUT_DIR / 'viz'          # Kaggle output — downloadable / 캐글 출력, 내려받을 수 있다
VIZ.mkdir(parents=True, exist_ok=True)
LIN_COLS = ('gen', 'uid', 'p1', 'p2', 'op', 'fit', 'gcv', 'nodes', 'n_genes')
LIN_TYPES = (np.int16, np.int32, np.int32, np.int32, np.uint8, np.float32, np.float32, np.int32, np.int16)


def sol_record(s):
    '''What prediction needs from the decider's solution, and nothing more.
    결정자 해에서 예측에 필요한 것만 옮긴다.'''
    return dict(intercept=float(s['intercept']), beta=np.asarray(s['beta'], float),
                mu=np.asarray(s['mu'], float), sd=np.asarray(s['sd'], float),
                alive=np.asarray(s['alive'], bool), g_lo=np.asarray(s['g_lo'], float),
                g_hi=np.asarray(s['g_hi'], float), alpha_j=np.asarray(s['alpha_j'], float),
                n_off=int(s.get('n_off', 0)), press=float(s['press_rmse']), gcv=float(s['gcv_rmse']),
                train=float(s['train_rmse']))


def run_fold(f):
    _t = time.time()
    d = DES[f]
    top_rows, cen_rows, exp_rows, champs = [], [], [], {}
    lin, ops = [], {}

    def on_gen(gen, top, vs, vb, extra):
        '''Called at the end of every generation. The champion is kept as a candidate for the pooled-validation
        rule (section 5); everything else is what the figures are drawn from.
        세대가 끝날 때마다 호출된다. 1등은 통합 valid 최적(5절)의 후보로 남기고, 나머지는 그림 재료로 쌓는다.'''
        for rank, (ind, v) in enumerate(zip(top, vs), 1):
            s = ind.sol or {}
            top_rows.append(dict(
                gen=gen, rank=rank, uid=ind.uid,
                p1=(ind.parents[0] if ind.parents else -1),
                p2=(ind.parents[1] if len(ind.parents) > 1 else -1), op=ind.op,
                press=s.get('press_rmse', np.nan), gcv=s.get('gcv_rmse', np.nan),
                train=s.get('train_rmse', np.nan), valid=v, nodes=ind.nodes, n_genes=len(ind.genes),
                alive=int(np.asarray(s['alive']).sum()) if s else 0, n_off=int(s.get('n_off', 0)) if s else 0,
                expr=' | '.join(str(g) for g in ind.genes)))
        c = top[0]
        champs[gen] = dict(gen=gen, uid=c.uid, op=c.op, parents=list(c.parents), valid=float(vs[0]),
                           nodes=c.nodes, genes=[str(g) for g in c.genes], sol=sol_record(c.sol))
        cen = (extra or {}).get('census')
        if cen:
            n_ind = max(1, cen['n_ind'])
            for kind, cnt in (('terminal', cen['terminals']), ('prim', cen['prims'])):
                tot = max(1, sum(cnt.values()))
                for name, n in cnt.most_common():
                    cen_rows.append(dict(gen=gen, kind=kind, name=name, count=n,
                                         share=n / tot, per_ind=n / n_ind))
            for r, (e, n) in enumerate(cen['exprs'], 1):
                exp_rows.append(dict(gen=gen, rank=r, count=n,
                                     share=n / max(1, cen['n_genes']), expr=e))
        edges = (extra or {}).get('lineage')
        if edges:
            # 개체군 전체의 간선을 세대마다 바로 정수·실수 배열로 줄인다(본 런은 fold당 300만 간선이다)
            cols = list(zip(*edges))
            cols[4] = [ops.setdefault(o, len(ops)) for o in cols[4]]
            lin.append([np.asarray(x, t) for x, t in zip(cols, LIN_TYPES)])

    res = run(d, dict(CFG), ckpt=on_gen)

    # Write the records as Kaggle output / 기록을 캐글 출력으로 저장한다
    pd.DataFrame(res['rows']).to_csv(VIZ / f'generations_fold{f}.csv', index=False)
    pd.DataFrame(top_rows).to_csv(VIZ / f'top_fold{f}.csv', index=False)
    pd.DataFrame(cen_rows).to_csv(VIZ / f'census_fold{f}.csv', index=False)
    pd.DataFrame(exp_rows).to_csv(VIZ / f'genes_fold{f}.csv', index=False)
    if lin:
        np.savez_compressed(VIZ / f'lineage_fold{f}.npz',
                            **{k: np.concatenate([g[j] for g in lin]) for j, k in enumerate(LIN_COLS)},
                            ops=np.array(sorted(ops, key=ops.get)))
    gs = res['grow_stop']
    return dict(fold=f, rows=res['rows'], champs=champs, grow_stop=(None if gs is None else int(gs)),
                stop=res['stop'], ma_gen=(int(res['ma_best'][1]) if res['ma_best'] is not None else None),
                n_gen=len(res['rows']), n_lineage=int(sum(len(g[0]) for g in lin)),
                y_lo=float(d.y_train.min()), y_hi=float(d.y_train.max()),
                secs=time.time() - _t, rss=RSS_MB())


_t0 = time.time()
RES = {}
with ProcessPoolExecutor(max_workers=PROCS, mp_context=mp.get_context('fork')) as ex:
    for r in ex.map(run_fold, range(5)):
        RES[r['fold']] = r
        _rows = pd.DataFrame(r['rows'])
        _b = _rows.loc[_rows.best_valid.idxmin()]
        print(f'  fold {r["fold"]}: grown at gen {r["grow_stop"] if r["grow_stop"] is not None else "-":>3} '
              f'· ended gen {r["n_gen"] - 1:>3} ({r["stop"]}) · best valid {_b.best_valid:.4f} '
              f'(gen {int(_b.gen)}, {int(_b.best_genes)} genes) · lineage {r["n_lineage"]:,} edges '
              f'· {r["secs"]/60:.1f} min · {r["rss"]:.0f} MB', flush=True)

print(f'\n*** {(time.time()-_t0)/60:.1f} min · five folds done ***')
print(f'records -> {VIZ}  ({sum(1 for _ in VIZ.iterdir())} files, '
      f'{sum(p.stat().st_size for p in VIZ.iterdir())/1e6:.1f} MB)')

  fold 0: grown at gen  49 · ended gen 299 (ramp_done) · best valid 0.1352 (gen 57, 37 genes) · lineage 3,000,000 edges · 404.9 min · 508 MB
  fold 1: grown at gen  99 · ended gen 325 (time) · best valid 0.1119 (gen 92, 45 genes) · lineage 3,260,000 edges · 600.2 min · 572 MB
  fold 2: grown at gen  79 · ended gen 329 (ramp_done) · best valid 0.1109 (gen 282, 14 genes) · lineage 3,300,000 edges · 524.8 min · 559 MB
  fold 3: grown at gen  59 · ended gen 309 (ramp_done) · best valid 0.1126 (gen 38, 35 genes) · lineage 3,100,000 edges · 462.0 min · 531 MB
  fold 4: grown at gen  58 · ended gen 308 (ramp_done) · best valid 0.0992 (gen 114, 20 genes) · lineage 3,090,000 edges · 476.4 min · 536 MB

*** 600.2 min · five folds done ***
records -> /kaggle/working/outputs/viz  (25 files, 205.8 MB)


### The pooled-validation rule / 통합 valid 최적

돌려줄 세대는 다섯 fold가 모두 끝난 뒤 한 번에 고른다. 이렇게 고르는 이유는 0절에 적었다.

1. fold마다 다 자람 판정 세대 T를 찾고, 판정 뒤 p세대째마다 다섯 fold의 'T + p세대' 1등이 낸 valid 제곱오차를 합쳐 RMSE 하나를 낸다.
2. 이 곡선을 11칸 중심 이동평균하고 가장 낮은 p 하나를 고른다. p 후보는 다섯 fold 모두 기록이 있는 범위까지다.
3. 다섯 fold 모두 'T + p세대'의 1등을 돌려준다. 그 1등의 유전자와 결정자 해는 학습 중에 세대마다 저장해 두었다.

어떤 fold가 판정 전에 시간 상한에 걸리면 이 규칙을 쓸 수 없다.<br>
그때는 fold마다 자기 valid 오차의 11세대 이동평균이 가장 낮은 세대를 돌려주고, 그 사실을 눈에 띄게 출력한다.

In [20]:
# The pooled-validation rule (통합 valid 최적): one common step after the growth stop for all five folds.
# 통합 valid 최적. 다섯 fold에 공통으로 쓸 '판정 + p세대' 하나를 고른다. 다섯 fold가 모두 끝난 뒤 한 번만 한다.
def pooled_curve(res, smooth):
    '''Offsets p after each fold's growth stop, the pooled validation RMSE at p, and its centred moving average.
    판정 뒤 위치 p마다 다섯 fold의 valid 제곱오차를 합친 RMSE와 그 중심 이동평균. 쓸 수 없으면 None.'''
    T = {f: r['grow_stop'] for f, r in res.items()}
    if any(t is None for t in T.values()):
        return None
    V = {f: pd.DataFrame(r['rows']).set_index('gen').best_valid for f, r in res.items()}
    N = {f: len(DES[f].y_valid) for f in res}
    hi = min(int(V[f].index.max()) - T[f] for f in res)
    if hi < 0:
        return None
    offs = np.arange(0, hi + 1)
    total = float(sum(N.values()))
    curve = pd.Series([np.sqrt(sum(N[f] * V[f][T[f] + p] ** 2 for f in res) / total) for p in offs], index=offs)
    ma = curve.rolling(int(smooth), center=True, min_periods=int(smooth) // 2 + 1).mean()
    return T, curve, ma


def champion_predict(f, ch):
    '''The champion's validation and test predictions from its saved genes and decider solution, clipped to
    the fold's training price range — the same ruler the run used for its validation score.
    저장한 유전자와 결정자 해로 valid·test를 예측하고 fold의 train 가격 범위로 자른다. 진화 중에 valid 점수를 잴 때와 같은 기준이다.'''
    d, r = DES[f], RES[f]
    pset = make_pset(d.arg_names)
    table = node_table(pset, d.n_cols)
    genes = [gp.PrimitiveTree.from_string(s, pset) for s in ch['genes']]
    p_va = predict(gene_matrix(Evaluator(d.X_valid, d.codes_valid or {}, table), genes, {}), ch['sol'])
    p_te = predict(gene_matrix(Evaluator(d.X_test, d.codes_test or {}, table), genes, {}), ch['sol'])
    return np.clip(p_va, r['y_lo'], r['y_hi']), np.clip(p_te, r['y_lo'], r['y_hi'])


POOL = pooled_curve(RES, CFG['return_smooth'])
PICK = {}
if POOL is not None:
    T, CURVE, MA = POOL
    P_BEST = int(MA.idxmin())
    for f in range(5):
        PICK[f] = dict(RES[f]['champs'][T[f] + P_BEST], rule='pooled', offset=P_BEST)
    print(f'pooled validation: offsets 0..{len(CURVE) - 1} after the growth stop · picked +{P_BEST} '
          f'· pooled valid {CURVE[P_BEST]:.4f} (moving average {MA[P_BEST]:.4f}, '
          f'curve {CURVE.min():.4f} ~ {CURVE.max():.4f})')
    pd.DataFrame(dict(offset=CURVE.index, pooled_valid=CURVE.values, pooled_ma=MA.values)) \
        .to_csv(VIZ / 'pooled.csv', index=False)
else:
    # A fold stopped on time before it was grown — no common step exists. Each fold falls back to its own
    # 11-generation moving average. This is loud on purpose.
    # 판정 전에 시간 상한으로 멈춘 fold가 있어 공통 위치를 정할 수 없다. fold마다 자기 valid 오차의 11세대 이동평균이 가장 낮은 세대로 대신한다.
    print('!!! pooled validation is not possible — every fold falls back to its own 11-generation moving average !!!')
    for f in range(5):
        PICK[f] = dict(RES[f]['champs'][RES[f]['ma_gen']], rule='per-fold MA11', offset=None)
    pd.DataFrame(dict(offset=[], pooled_valid=[], pooled_ma=[])).to_csv(VIZ / 'pooled.csv', index=False)

for f in range(5):
    PICK[f]['pred_valid'], PICK[f]['pred_test'] = champion_predict(f, PICK[f])
    _ch = PICK[f]
    print(f'  fold {f}: gen {_ch["gen"]:>3} · {int(_ch["sol"]["alive"].sum())} genes alive '
          f'({_ch["sol"]["n_off"]} switched off by ARD) · nodes {_ch["nodes"]:>4} · valid {_ch["valid"]:.4f} '
          f'· PRESS {_ch["sol"]["press"]:.4f}')

pooled validation: offsets 0..226 after the growth stop · picked +1 · pooled valid 0.1181 (moving average 0.1178, curve 0.1174 ~ 0.1277)
  fold 0: gen  50 · 33 genes alive (3 switched off by ARD) · nodes  216 · valid 0.1388 · PRESS 0.0953
  fold 1: gen 100 · 44 genes alive (2 switched off by ARD) · nodes  417 · valid 0.1151 · PRESS 0.0917
  fold 2: gen  80 · 45 genes alive (7 switched off by ARD) · nodes  338 · valid 0.1141 · PRESS 0.0941
  fold 3: gen  60 · 44 genes alive (10 switched off by ARD) · nodes  309 · valid 0.1187 · PRESS 0.0986
  fold 4: gen  59 · 44 genes alive (9 switched off by ARD) · nodes  299 · valid 0.1006 · PRESS 0.1014


# 6. Validation / 검증

확인하는 것은 세 가지다.

1. **재적합** : 고른 1등의 유전자로 ARD를 다시 풀어 같은 계수, 같은 valid 점수가 나오는지 본다. 결정자 해를 옮기다 깨진 곳이 있으면 여기서 드러난다.
2. **기록 대조** : 그 점수가 진화 중에 기록한 그 세대 1등의 valid 점수와 같은지 본다. 고른 세대와 돌려준 모델이 같은지 확인하는 단계다.
3. **제출 형식** : 행 수, `sample_submission`과 같은 Id 순서, 값이 모두 유한한지, 가격 범위가 정상인지 본다.

fold valid 점수를 리더보드 점수의 추정치로 내세우지 않는다 — valid 행은 돌려줄 세대를 고르는 데 이미 썼으므로 더는 편향 없는 추정치가 아니다.<br>
공정한 평가는 리더보드가 한다. 다만 valid 점수가 기록에서 크게 벗어나면 제출을 막는 기준으로는 쓴다(8절의 점수 문턱).

In [21]:
# Check 2 of 4 — do the returned genes reproduce the returned model and its validation score?
# 검증 2/4 — 반환된 유전자가 반환된 모델과 그 valid 점수를 재현하는가?
# The training log-price range is the yardstick for "is this prediction plausible".
# '이 예측이 그럴듯한가'는 train log 가격 범위로 판단한다.
Y_LO = float(min(RES[f]['y_lo'] for f in range(5)))
Y_HI = float(max(RES[f]['y_hi'] for f in range(5)))
print(f'training log-price range [{Y_LO:.3f}, {Y_HI:.3f}] '
      f'= ${np.exp(Y_LO):,.0f} ~ ${np.exp(Y_HI):,.0f}\n')

sub_head = pd.read_csv(COMP_DIR / 'sample_submission.csv', nrows=0)
assert list(sub_head.columns) == ['Id', 'SalePrice'], 'sample_submission columns changed!!'
sample_ids = pd.read_csv(COMP_DIR / 'sample_submission.csv').Id.to_numpy()

for f in range(5):
    ch, d, r = PICK[f], DES[f], RES[f]
    pset = make_pset(d.arg_names)
    table = node_table(pset, d.n_cols)
    genes = [gp.PrimitiveTree.from_string(s, pset) for s in ch['genes']]
    # Refit ARD from the genes alone — it must land on the same coefficients and the same validation score.
    # 저장한 계수를 쓰지 않고 유전자만으로 ARD를 다시 풀어 같은 계수, 같은 valid 점수가 나오는지 본다.
    sol = fit_ard(gene_matrix(Evaluator(d.X_train, d.codes_train or {}, table), genes, {}), d.y_train)
    rv_pred = predict(gene_matrix(Evaluator(d.X_valid, d.codes_valid or {}, table), genes, {}), sol)
    rv = float(np.sqrt(np.mean((np.clip(rv_pred, r['y_lo'], r['y_hi']) - d.y_valid) ** 2)))
    pv = ch['pred_valid']
    got = float(np.sqrt(np.mean((pv - d.y_valid) ** 2)))
    assert np.isfinite(pv).all(), f'fold {f} valid prediction has non-finite values!!'
    assert abs(got - ch['valid']) < 1e-9, \
        f'fold {f} returned valid {got:.6f} != the run\'s record for gen {ch["gen"]} ({ch["valid"]:.6f})!!'
    assert np.abs(sol['beta'] - ch['sol']['beta']).max() < 1e-6, f'fold {f} refit coefficients differ!!'
    assert abs(rv - ch['valid']) < 1e-6, f'fold {f} refit from genes gives {rv:.6f}, returned {ch["valid"]:.6f}!!'

    # The verified model goes to the output too — genes and their ARD weights are what the figures read.
    # 검증을 통과한 모델도 출력으로 남긴다. 그림에 쓰는 것은 유전자와 그 ARD 계수다.
    _fin = lambda a: [float(x) if np.isfinite(x) else None for x in np.asarray(a, float)]
    (VIZ / f'model_fold{f}.json').write_text(json.dumps(dict(
        fold=f, rule=ch['rule'], gen=int(ch['gen']), offset=ch['offset'], grow_stop=r['grow_stop'],
        uid=int(ch['uid']), valid=float(ch['valid']), press=ch['sol']['press'], n_gen=int(r['n_gen']),
        intercept=ch['sol']['intercept'], beta=_fin(ch['sol']['beta']), alpha_j=_fin(ch['sol']['alpha_j']),
        alive=[bool(a) for a in ch['sol']['alive']], n_off=ch['sol']['n_off'],
        mu=_fin(ch['sol']['mu']), sd=_fin(ch['sol']['sd']), g_lo=_fin(ch['sol']['g_lo']), g_hi=_fin(ch['sol']['g_hi']),
        y_lo=r['y_lo'], y_hi=r['y_hi'], genes=list(ch['genes']), nodes=int(ch['nodes'])), indent=1), encoding='utf-8')

    pt = ch['pred_test']
    assert len(pt) == len(sample_ids) and np.isfinite(pt).all(), f'fold {f} test prediction!!'
    assert d.ids_test.tolist() == sample_ids.tolist(), f'fold {f} test Id order != sample_submission!!'
    assert Y_LO <= float(np.median(pt)) <= Y_HI, \
        f'fold {f} median prediction {np.median(pt):.2f} is outside the training price range!!'
    print(f'  fold {f}: gen {ch["gen"]} · valid {ch["valid"]:.4f} · refit from genes {rv:.4f} '
          f'· test range [{pt.min():.2f}, {pt.max():.2f}] · {r["n_gen"]} generations logged')

print('\n*** Validation passed — genes reproduce the model and its score, Id order matches ***')

training log-price range [10.460, 13.534] = $34,900 ~ $755,000

  fold 0: gen 50 · valid 0.1388 · refit from genes 0.1388 · test range [10.46, 13.42] · 300 generations logged
  fold 1: gen 100 · valid 0.1151 · refit from genes 0.1151 · test range [10.46, 13.41] · 326 generations logged
  fold 2: gen 80 · valid 0.1141 · refit from genes 0.1141 · test range [10.60, 13.29] · 330 generations logged
  fold 3: gen 60 · valid 0.1187 · refit from genes 0.1187 · test range [10.65, 13.52] · 310 generations logged
  fold 4: gen 59 · valid 0.1006 · refit from genes 0.1006 · test range [10.54, 13.39] · 309 generations logged

*** Validation passed — genes reproduce the model and its score, Id order matches ***


# 7. Records for the figures / 그림에 쓸 기록

학습 중에 세대마다 쌓은 기록을 파일로 남긴다. 진화를 다시 돌리지 않고도 궤적 · 계보 · 반환 유전자를 볼 수 있게 하기 위해서다.<br>
fold마다 파일 여섯 개와 통합 valid 최적 곡선 파일 하나가 `outputs/viz/`에 저장되고, 이 절은 빠진 파일이 없는지 확인한다.

| 파일 | 내용 | 쓰임 |
| --- | --- | --- |
| `generations_fold{f}.csv` | 세대별 1등 지표(PRESS · valid · 유전자 수 · 꺼진 유전자)와 개체군 통계, 유전자 가격, 다 자람 판정 | 궤적, 분포 부채꼴 |
| `top_fold{f}.csv` | 세대마다 상위 5개의 `uid` · 부모 · 연산자 · 유전자식 | 세대별 1등, 유전자 유지 / 탈락 / 신규 |
| `lineage_fold{f}.npz` | 개체군 전체의 부모-자식 간선 | 계보 그래프 |
| `census_fold{f}.csv` | 터미널과 연산자의 세대별 점유율 | 유전자 잔존율 |
| `genes_fold{f}.csv` | 세대마다 가장 흔한 유전자식 50개 | 식 단위의 유행과 소멸 |
| `model_fold{f}.json` | 반환 모델 — 유전자, ARD 계수와 유전자별 α, 유전자 값 범위(6절에서 검증한 뒤 저장) | 유전자 영향력 |
| `pooled.csv` | 통합 valid 최적 곡선 — 위치 p, 합친 valid 오차, 이동평균 | 반환 위치 |

계보는 본 런에서 fold당 300만 간선 안팎이라 압축 npz로, 나머지는 CSV로 둔다.<br>
확인이 끝나면 파일 31개를 `viz_records.zip` 하나로 묶는다 — Kaggle 제출 화면은 출력 폴더의 파일을 알파벳 순으로 보여 주므로, 낱개 파일이 많으면 `submission.csv`가 맨 뒤로 밀린다.

In [22]:
# The records let the run be read without re-running it. All this cell does is confirm nothing is missing.
# 진화를 다시 돌리지 않고도 런을 살펴볼 수 있게 남긴 기록이다. 이 셀은 빠진 파일이 없는지만 확인한다.
_want = ('generations', 'top', 'census', 'genes', 'model') + (('lineage',) if CFG['lineage'] else ())
_ext = {'lineage': 'npz', 'model': 'json'}

print(f'{"fold":>5}' + ''.join(f'{_n:>14}' for _n in _want))
_total = 0
for f in range(5):
    _cells = []
    for _n in _want:
        _p = VIZ / f'{_n}_fold{f}.{_ext.get(_n, "csv")}'
        assert _p.exists(), f'{_p.name} is missing — the records are incomplete!!'
        assert _p.stat().st_size > 0, f'{_p.name} is empty!!'
        _total += _p.stat().st_size
        _cells.append(f'{_p.stat().st_size / 1024:>10,.0f} KB')
    print(f'{f:>5}' + ''.join(f'{_c:>14}' for _c in _cells))
assert (VIZ / 'pooled.csv').exists(), 'pooled.csv is missing!!'

# The population statistics, the price schedule and the lineage columns are what the figures need — a wrong
# CFG would write the files anyway, just without them.
# 그림에 필요한 것은 개체군 통계·가격 일정·계보 열이다. CFG가 틀려도 파일은 만들어지고 이 열들만 빠지므로 따로 확인한다.
_gen = pd.read_csv(VIZ / 'generations_fold0.csv')
assert {'best_press', 'best_valid', 'pop_min_press', 'gcv_q05', 'gcv_q95', 'gene_price', 'grow_stop',
        'avg_genes', 'best_off'} <= set(_gen.columns), 'the generation log is missing columns!!'
_top = pd.read_csv(VIZ / 'top_fold0.csv')
assert {'uid', 'p1', 'p2', 'op', 'expr'} <= set(_top.columns), 'the top-k log is missing lineage columns!!'
assert _top.expr.notna().all(), 'a top-k row has no expression!!'

print(f'\n*** Records complete — {_total / 1e6:.1f} MB in {VIZ.name}/ ***')
print(f'    model_fold{{f}}.json carries the returned genes and ARD weights; lineage_fold{{f}}.npz the parent-child edges')
print(f'    generations {len(_gen):,} rows · top-k {len(_top):,} rows (fold 0)')

# Kaggle's "Submit to Competition -> Notebook" list is the output tree in alphabetical order, so 31
# loose files under outputs/viz/ push submission.csv to the very bottom. One zip keeps the root short.
# 캐글 제출 목록은 출력 폴더의 파일을 알파벳 순으로 보여 준다. outputs/viz/ 아래 낱개 파일 31개가 있으면
# submission.csv가 맨 아래로 밀린다. 확인이 끝나면 하나로 묶고, 이름이 's' 뒤에 오게 해 submission.csv가 맨 위에 보이게 한다.
_zip = ROOT_DIR / 'viz_records.zip'
with zipfile.ZipFile(_zip, 'w', zipfile.ZIP_DEFLATED) as _z:
    for _p in sorted(VIZ.iterdir()):
        _z.write(_p, _p.name)      # 디렉터리 없이 파일 이름만 넣는다 / file names only, no paths
shutil.rmtree(OUTPUT_DIR)

print(f'\n*** Packed -> {_zip.name}  {_zip.stat().st_size / 1e6:.1f} MB (from {_total / 1e6:.1f} MB) ***')
print(f'    output root: submission.csv, {_zip.name} — submission.csv sorts first')
print(f'    출력 루트에는 submission.csv와 {_zip.name}뿐이고, submission.csv가 먼저 온다')

 fold   generations           top        census         genes         model       lineage
    0        164 KB      2,251 KB      1,637 KB      1,933 KB          7 KB     29,477 KB
    1        178 KB      4,897 KB      1,792 KB      2,514 KB         10 KB     34,783 KB
    2        180 KB      3,630 KB      1,807 KB      2,365 KB         10 KB     35,009 KB
    3        169 KB      2,683 KB      1,694 KB      2,186 KB          9 KB     32,758 KB
    4        169 KB      2,721 KB      1,688 KB      2,198 KB          9 KB     32,059 KB

*** Records complete — 205.8 MB in viz/ ***
    model_fold{f}.json carries the returned genes and ARD weights; lineage_fold{f}.npz the parent-child edges
    generations 300 rows · top-k 1,500 rows (fold 0)

*** Packed -> viz_records.zip  151.4 MB (from 205.8 MB) ***
    output root: submission.csv, viz_records.zip — submission.csv sorts first
    출력 루트에는 submission.csv와 viz_records.zip뿐이고, submission.csv가 먼저 온다


# 8. Submission / 제출

### Score gate / 점수 문턱

학습과 기록 저장은 이미 끝났고, 로그와 `viz_records.zip`도 남아 있다. 여기서 막는 것은 제출뿐이다.

fold valid 점수는 돌려줄 세대를 고르는 데 쓴 값이라 리더보드 점수를 예측하지는 못한다.<br>
그래도 기록과 나란히 놓으면 런이 망가졌는지는 알 수 있다 — 실행 환경이 다르면 수치가 조금 움직이겠지만, 크게 움직여서는 안 된다.

1. **기록 문턱** : fold 5개의 valid 평균이 기록 0.1229보다 0.010 이상 나쁘면 제출을 막는다. 기록은 같은 셋업을 같은 fold 표로 끝까지 돌린 런의 값이다.
2. **기준선 문턱** : 같은 fold에서 릿지 89열을 이기지 못하면 제출을 막는다. 기준선은 4절에서 이 런이 직접 잰 값이라, 분할 난이도가 달라져도 기준이 흔들리지 않는다. 기록은 이 기준선보다 평균 0.0061 앞섰다.

0.010은 이전 런의 점수 문턱과 같은 폭이다.<br>
기록과 같은 런에서 반환 규칙만 바꾸면 0.1148\~0.1253이 나오고, 결정자만 α 하나짜리 릿지로 바꾼 런은 0.1238이다.<br>
이번 분할에서는 릿지 89열이 0.1290이라 기준선 문턱이 먼저 걸린다. QUICK 런에서는 수치만 출력하고 제출을 막지 않는다.

In [23]:
# Check 3 of 4 — the score gate. Training and the records are already done; only the submission is gated.
# 검증 3/4 — 점수 문턱. 학습과 기록은 이미 끝났고, 여기서 막을 수 있는 것은 제출뿐이다.
# REF_VALID is the one number that comes from outside this notebook: an earlier run's score, which
# the data cannot tell you. The ridge baseline is measured here instead of quoted.
# 이 노트북 밖에서 가져온 값은 REF_VALID 하나뿐이다. 앞서 돌린 런의 점수라 데이터에서 계산할 수 없다. 릿지 기준선은
# 옮겨 오지 않고 4절에서 직접 쟀다.
REF_VALID = 0.1229      # the record: the same setup run to the end, mean valid of the 5 returned models
                        # 기록 — 같은 셋업을 끝까지 돌린 런, 반환 모델 5개의 valid 평균
REF_MARGIN = 0.0061     # how far that run beat ridge-89 on its own folds / 그 런이 릿지 89열을 이긴 폭
REF_TOL = 0.010         # the same record run gives 0.1148~0.1253 under other return rules, and 0.1238 with a single-alpha
                        # ridge; ridge-89 sits +0.0061 away, so the baseline gate binds first
                        # 같은 런에서 반환 규칙만 바꾸면 0.1148~0.1253, α 하나짜리 릿지로 바꾸면 0.1238이다.
                        # 릿지 89열은 +0.0061 밖이라 기준선 문턱이 먼저 걸린다

gp_valid = np.array([PICK[f]['valid'] for f in range(5)])
ridge = np.array([RIDGE89[f] for f in range(5)])

print(f'{"fold":>5}{"GP valid":>10}{"ridge-89":>10}{"margin":>9}')
for f in range(5):
    print(f'{f:>5}{gp_valid[f]:>10.4f}{ridge[f]:>10.4f}{ridge[f] - gp_valid[f]:>+9.4f}')

gap = float(gp_valid.mean() - REF_VALID)
margin = float(ridge.mean() - gp_valid.mean())
print(f'\nmean GP valid {gp_valid.mean():.4f} · record {REF_VALID:.4f} · gap {gap:+.4f} (tolerance {REF_TOL})')
print(f'margin over ridge-89 on these folds {margin:+.4f} · record margin {REF_MARGIN:+.4f}')
print(f'returned by the {PICK[0]["rule"]} rule')

if QUICK:
    GATE_OK = True
    print('\nQUICK run — the gate is reported, not enforced. The submission below is a wiring check, not a result.')
    print('QUICK 런 — 문턱 결과는 출력만 하고 제출을 막지 않는다. 아래 제출 파일은 동작 확인용이며 실제 결과가 아니다.')
else:
    assert gap < REF_TOL, \
        f'mean valid {gp_valid.mean():.4f} is {gap:+.4f} off the record — significantly worse, submission blocked!!'
    assert margin > 0, \
        f'GP does not beat ridge-89 on its own folds (margin {margin:+.4f}) — submission blocked!!'
    GATE_OK = True
    print('\n*** Score gate passed — within tolerance of the record ***')

 fold  GP valid  ridge-89   margin
    0    0.1388    0.1384  -0.0004
    1    0.1151    0.1329  +0.0178
    2    0.1141    0.1293  +0.0152
    3    0.1187    0.1237  +0.0050
    4    0.1006    0.1208  +0.0202

mean GP valid 0.1174 · record 0.1229 · gap -0.0055 (tolerance 0.01)
margin over ridge-89 on these folds +0.0116 · record margin +0.0061
returned by the pooled rule

*** Score gate passed — within tolerance of the record ***


### Write submission.csv / 제출 파일 저장

In [24]:
# Check 4 of 4 — the submission contract. The ensemble is the mean of the five fold models in log space.
# 검증 4/4 — 제출 형식. 앙상블은 fold 모델 5개의 예측을 log 공간에서 평균한 값이다.
assert GATE_OK, 'the score gate has not passed — submission blocked!!'

P = np.array([PICK[f]['pred_test'] for f in range(5)])
ens = P.mean(axis=0)

assert P.shape == (5, len(sample_ids)), f'prediction block {P.shape}!!'
assert np.isfinite(ens).all(), 'non-finite values in the ensemble!!'
# Each fold is already clipped to its own training range, so the mean cannot leave the overall one.
# fold마다 자기 train 범위로 이미 잘랐으니 평균은 전체 범위를 벗어날 수 없다.
assert (ens >= Y_LO).all() and (ens <= Y_HI).all(), \
    f'ensemble leaves the training price range [{ens.min():.2f}, {ens.max():.2f}]!!'

submission = pd.DataFrame({'Id': sample_ids, 'SalePrice': np.exp(ens)})

assert list(submission.columns) == ['Id', 'SalePrice'], 'submission columns != contract!!'
assert len(submission) == len(sample_ids), 'row count mismatch!!'
assert submission.notna().all().all(), 'NaN detected in submission!!'
assert submission.SalePrice.gt(0).all(), 'non-positive price!!'

submission.to_csv(SUB_PATH, index=False)
print(f'fold valid    : ' + ', '.join(f'{PICK[f]["valid"]:.4f}' for f in range(5)))
print(f'spread between fold predictions (mean sd): {P.std(axis=0).mean():.4f}')
print(f'*** Submission written -> {SUB_PATH}  {len(submission):,} rows ***')
print(f'    SalePrice [{submission.SalePrice.min():,.0f}, {submission.SalePrice.max():,.0f}] '
      f'median {submission.SalePrice.median():,.0f}')

fold valid    : 0.1388, 0.1151, 0.1141, 0.1187, 0.1006
spread between fold predictions (mean sd): 0.0364
*** Submission written -> /kaggle/working/submission.csv  1,459 rows ***
    SalePrice [44,357, 607,938] median 157,144


In [25]:
submission.head()

,Id,SalePrice
0,1461,118031.263250
1,1462,157576.785136
2,1463,182185.308287
3,1464,195891.813370
4,1465,179440.726342


In [26]:
print('*** Done. ***')

*** Done. ***


# 끝.